In [3]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask
import glob
import pickle
import os
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, time
from datetime import timedelta

In [4]:
#REQUIRED FOR ALL
# === 1. Load SPOT CSV with Datetime column ===
spot_df = pd.read_csv(
    "/home/newberry3/main/_NIFTY_IDX__202507041318.csv",
    usecols=["Date", "Time", "Open", "High", "Low", "Close"]
)

# Combine Date & Time into a Datetime column
spot_df["Datetime"] = pd.to_datetime(
    spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str), 
    errors='coerce'
)
num_bad_spot = spot_df["Datetime"].isna().sum()
if num_bad_spot > 0:
    print(f"⚠️  Dropping {num_bad_spot} bad rows from spot_df due to unparseable Datetime.")
    spot_df = spot_df.dropna(subset=["Datetime"])

spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

print("spot_df preview:")
print(spot_df.head())

# Save and reload to Parquet (optional)
try:

    print("spot_df loaded successfully from parquet.")
except Exception as e:
    print(f"Error saving/loading spot_df: {e}")

#load spot data

# --- 2. Ensure Datetime is datetime and sort for time-based ops ---
spot_df['Datetime'] = pd.to_datetime(spot_df['Datetime'])
spot_df = spot_df.sort_values('Datetime')

# --- 3. Set Datetime as index for resampling ---
spot_df = spot_df.set_index('Datetime')

# --- 4. Filter to regular NIFTY trading hours (avoid pre/post-market ticks) ---
spot_df = spot_df.between_time('09:15:00', '15:25:00')

# --- 5. Resample to 60-min OHLCV bars. Offset=15min for NIFTY standard (9:15 open) ---
spot_1min = spot_df.resample(
    '1min', 
    origin='start_day', 
    offset='15min', 
    label='left', 
    closed='left'
).agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()

# --- 6. Reset index back to columns for easier future ops ---
spot_1min = spot_1min.reset_index()


# --- 9. Preview final 60-min OHLC + EMA DataFrame ---
print(spot_1min.tail())

spot_df preview:
             Datetime      Open      High       Low     Close
0 2024-05-28 13:57:00  22933.20  22934.35  22926.00  22928.75
1 2024-05-28 13:58:00  22929.05  22933.15  22928.15  22929.60
2 2024-05-28 13:59:00  22929.85  22933.40  22927.50  22929.05
3 2024-05-28 14:00:00  22929.20  22929.55  22921.40  22921.90
4 2024-05-28 14:01:00  22921.50  22931.85  22917.85  22928.20
spot_df loaded successfully from parquet.
                  Datetime      Open      High       Low     Close
369690 2025-06-13 15:21:00  24720.50  24722.45  24718.20  24721.40
369691 2025-06-13 15:22:00  24721.05  24728.85  24720.75  24727.45
369692 2025-06-13 15:23:00  24726.70  24728.90  24725.95  24728.10
369693 2025-06-13 15:24:00  24728.15  24733.15  24727.40  24729.35
369694 2025-06-13 15:25:00  24728.55  24734.40  24728.45  24733.55


In [5]:
# --- A. Add session_date for grouping (IST hours already trimmed) ---
spot_1min["session_date"] = spot_1min["Datetime"].dt.date

# --- B. Build daily OHLC (from your 1-min bars) ---
daily = (
    spot_1min
    .groupby("session_date")
    .agg(High=("High","max"), Low=("Low","min"), Close=("Close","last"))
    .sort_index()
)

# OPTIONAL: drop incomplete days (missing enough minutes OR missing OR window)
# Keep sessions that have at least, say, 350 minutes (≈ full 09:15–15:25 is 371 mins)
counts = spot_1min.groupby("session_date")["Datetime"].count()
good_days = counts[counts >= 350].index
daily = daily.loc[good_days]

# --- C. Compute yesterday-based Pivot Range (PP, SV, PivotLow/High) ---
prev = daily.shift(1).rename(columns={"High":"Prev_H","Low":"Prev_L","Close":"Prev_C"})
D = daily.join(prev[["Prev_H","Prev_L","Prev_C"]])

D["PP"] = (D["Prev_H"] + D["Prev_L"] + D["Prev_C"]) / 3.0
D["SV"] = (D["Prev_H"] + D["Prev_L"]) / 2.0
D["PD"] = D["PP"] - D["SV"]
D["PivotLow"]  = D["PP"] - D["PD"].abs()
D["PivotHigh"] = D["PP"] + D["PD"].abs()

# --- D. Compute Opening Range per day (e.g., T=20 minutes => 09:15–09:34) ---
T = 20  # you can parameterize later
or_start = pd.to_datetime(spot_1min["Datetime"].dt.date.astype(str) + " 09:15:00")
or_end   = or_start + pd.Timedelta(minutes=T-1)
# attach these to each row for a clean mask
tmp = spot_1min.copy()
tmp["or_start"] = pd.to_datetime(tmp["session_date"].astype(str) + " 09:15:00")
tmp["or_end"]   = tmp["or_start"] + pd.Timedelta(minutes=T-1)

in_or = (tmp["Datetime"] >= tmp["or_start"]) & (tmp["Datetime"] <= tmp["or_end"])
or_table = (
    tmp[in_or]
    .groupby("session_date")
    .agg(ORH=("High","max"), ORL=("Low","min"))
    .reindex(D.index)  # align to daily index
)

# --- E. Rolling avgRange_n for a grid of n (lookback days), shifted to avoid look-ahead ---
D["DailyRange"] = D["High"] - D["Low"]
n_grid = list(range(12, 31, 2))  # tweak as you like

for n in n_grid:
    D[f"avgRange_{n}"] = (
        D["DailyRange"].rolling(n).mean().shift(1)
    )

# --- F. Merge OR with daily to get a compact "day metadata" table ---
meta = D.join(or_table)

# If any day lacks required fields (first n days, no prev day for pivots), drop them
need_cols = ["ORH","ORL","PP","PivotLow","PivotHigh"] + [f"avgRange_{n}" for n in n_grid]
meta = meta.dropna(subset=need_cols)

print("Daily meta preview:")
print(meta.head())


Daily meta preview:
                  High      Low     Close    Prev_H    Prev_L    Prev_C  \
session_date                                                              
2021-07-13    15820.80  15744.6  15812.90  15789.20  15644.75  15696.80   
2021-07-14    15877.35  15764.2  15837.30  15820.80  15744.60  15812.90   
2021-07-15    15952.35  15855.0  15922.60  15877.35  15764.20  15837.30   
2021-07-16    15962.25  15882.6  15922.95  15952.35  15855.00  15922.60   
2021-07-19    15836.90  15707.5  15758.10  15962.25  15882.60  15922.95   

                        PP         SV         PD   PivotLow  ...  avgRange_16  \
session_date                                                 ...                
2021-07-13    15710.250000  15716.975  -6.725000  15703.525  ...   132.412500   
2021-07-14    15792.766667  15782.700  10.066667  15782.700  ...   120.956250   
2021-07-15    15826.283333  15820.775   5.508333  15820.775  ...   119.050000   
2021-07-16    15909.983333  15903.675   6.308333 

In [ ]:
# # ===================== ACD + Pivot (paper-faithful) yearly backtest (TP = k × A) =====================
# import pandas as pd, numpy as np, os
# from datetime import timedelta

# # ======== USER INPUTS ========
# YEAR            = 2021        # year to trade
# T               = 20          # opening range minutes
# W               = 20          # walk-forward window (training days)
# n_grid          = list(range(12, 31, 2))            # avg-range lookbacks to scan (paper optimizes n)
# P_grid          = [x/100 for x in range(10,31,2)]   # A distance = P * avgRange_n
# slippage_pts    = 0.5
# cost_pts_rt     = 0.5         # round-trip cost (pts)
# target_mult_A   = 1.5         # TAKE-PROFIT = target_mult_A × A (vol-adjusted)
# fill_at_level   = True        # fill exactly at target/stop when touched
# allow_c_reversal= False       # allow a single C reversal after established A trade

# # ======== REQUIRE: spot_1min already built (Datetime, Open, High, Low, Close), 09:15–15:25 only ========

# # --- Year bounds + auto buffer ---
# year_start = pd.Timestamp(f"{YEAR}-01-01")
# year_end   = pd.Timestamp(f"{YEAR}-12-31")
# need_sessions = W + max(n_grid)   # prior sessions required before first tradable day

# buffer_days = 180                 # start with a larger buffer for a full year
# max_buffer_days = 540

# def build_meta_from_minutes(spot_slice: pd.DataFrame, T: int, n_grid: list[int]) -> pd.DataFrame:
#     """Daily meta: ORH/ORL, PivotRange (PP, SV, PD => PivotLow/High), shifted rolling avgRange_n."""
#     z = spot_slice.copy()
#     z["session_date"] = z["Datetime"].dt.date

#     # Daily OHLC from minutes
#     daily = (z.groupby("session_date")
#                .agg(High=("High","max"), Low=("Low","min"), Close=("Close","last"))
#                .sort_index())

#     # Drop incomplete days (OR window relies on near-full session)
#     counts = z.groupby("session_date")["Datetime"].count()
#     good_days = counts[counts >= 350].index
#     daily = daily.loc[good_days]

#     # Prior-day pivots
#     prev = daily.shift(1).rename(columns={"High":"Prev_H","Low":"Prev_L","Close":"Prev_C"})
#     D = daily.join(prev[["Prev_H","Prev_L","Prev_C"]])
#     D["PP"] = (D["Prev_H"] + D["Prev_L"] + D["Prev_C"]) / 3.0
#     D["SV"] = (D["Prev_H"] + D["Prev_L"]) / 2.0
#     D["PD"] = D["PP"] - D["SV"]
#     D["PivotLow"]  = D["PP"] - D["PD"].abs()
#     D["PivotHigh"] = D["PP"] + D["PD"].abs()

#     # Opening range per day (T minutes from 09:15 inclusive)
#     tmp = z.copy()
#     tmp["or_start"] = pd.to_datetime(tmp["session_date"].astype(str) + " 09:15:00")
#     tmp["or_end"]   = tmp["or_start"] + pd.Timedelta(minutes=T-1)
#     in_or = (tmp["Datetime"] >= tmp["or_start"]) & (tmp["Datetime"] <= tmp["or_end"])
#     or_table = (tmp[in_or].groupby("session_date").agg(ORH=("High","max"), ORL=("Low","min")).reindex(D.index))

#     # Shifted rolling daily range means (no look-ahead)
#     D["DailyRange"] = D["High"] - D["Low"]
#     for n in n_grid:
#         D[f"avgRange_{n}"] = D["DailyRange"].rolling(n).mean().shift(1)

#     meta_built = D.join(or_table)
#     need_cols = ["ORH","ORL","PP","PivotLow","PivotHigh"] + [f"avgRange_{n}" for n in n_grid]
#     return meta_built.dropna(subset=need_cols)

# # Auto-expand buffer until we have enough pre-history
# while True:
#     hist_start = year_start - pd.Timedelta(days=buffer_days)
#     spot_y = spot_1min[(spot_1min["Datetime"] >= hist_start) & (spot_1min["Datetime"] <= year_end)].copy()
#     meta_full = build_meta_from_minutes(spot_y, T=T, n_grid=n_grid)
#     if len(meta_full.index) > need_sessions:
#         break
#     buffer_days = min(buffer_days + 60, max_buffer_days)
#     if buffer_days >= max_buffer_days:
#         raise ValueError("Not enough pre-history even after max buffer; reduce W/max(n_grid) or extend data.")

# # Prepare minute access (session_date, Datetime index) using buffered slice
# minute_df = spot_y.copy()
# minute_df["session_date"] = minute_df["Datetime"].dt.date
# minute_df = minute_df.set_index(["session_date","Datetime"]).sort_index()

# def get_day_minutes(day):
#     try:
#         df = minute_df.loc[day][["Open","High","Low","Close"]].copy()
#         if not isinstance(df.index, pd.DatetimeIndex): df = df.set_index("Datetime")
#         return df
#     except KeyError:
#         return pd.DataFrame(columns=["Open","High","Low","Close"])

# # ---- A/C levels per paper: A = P * avgRange_n ; C = 2*A (distance from OR bounds) ----
# def compute_ac_levels(ORH, ORL, avg_range, P):
#     A = P * float(avg_range)
#     return {"A": A, "Aup": ORH + A, "Adown": ORL - A, "Cup": ORH + 2*A, "Cdown": ORL - 2*A}

# # A is established when Close stays beyond threshold for ≥ T/2 minutes
# def hold_established(series_close, threshold, side, hold_minutes):
#     if series_close.empty: return None
#     cond = (series_close >= threshold) if side=="long" else (series_close <= threshold)
#     groups = (~cond).cumsum()
#     run = cond.groupby(groups).cumcount() + 1
#     hits = run[(cond) & (run >= hold_minutes)]
#     return None if hits.empty else hits.index[0]

# def simulate_day(mins_day, row_meta, n, P, T=20, hold=None,
#                  slippage=0.0, cost_pts=0.0,
#                  target_mult_A=1.0, fill_at_level=True, allow_c_reversal=False):
#     """Paper rules with TP = target_mult_A × A:
#        - A established after hold=T/2 beyond Aup/Adown → enter.
#        - B/D stop refined by Pivot Range (long stop = max(ORL, PivotLow); short stop = min(ORH, PivotHigh)).
#        - C distance = 2*A. Optional single reversal at C with D stop (≈ OR bound).
#        - Profit-taking: TP = target_mult_A × A; else EOD flat.
#     """
#     if mins_day.empty: return []
#     hold = hold or (T//2)

#     ORH, ORL = float(row_meta["ORH"]), float(row_meta["ORL"])
#     Avals = compute_ac_levels(ORH, ORL, row_meta["avgRange_n"], P)
#     Aup, Adown, Cup, Cdown, Adist = Avals["Aup"], Avals["Adown"], Avals["Cup"], Avals["Cdown"], Avals["A"]

#     # A establishment (earliest)
#     t_est_long  = hold_established(mins_day["Close"], Aup,   "long",  hold)
#     t_est_short = hold_established(mins_day["Close"], Adown, "short", hold)

#     cands = []
#     if t_est_long  is not None:  cands.append(("long",  t_est_long))
#     if t_est_short is not None:  cands.append(("short", t_est_short))
#     if not cands: return []

#     cands.sort(key=lambda x: x[1])
#     side, t_entry = cands[0]
#     entry_px = float(mins_day.loc[t_entry, "Close"]) + (slippage if side=="long" else -slippage)

#     # Paper-refined B/D stops, TP = k × A
#     tgt_dist = target_mult_A * Adist
#     if side=="long":
#         stop = max(ORL, float(row_meta["PivotLow"]))
#         target = entry_px + tgt_dist
#         reverse_level = Cdown
#         reverse_side  = "short"
#         d_stop_level  = ORH  # ≈ ORH (+tick if modeling ticks)
#     else:
#         stop = min(ORH, float(row_meta["PivotHigh"]))
#         target = entry_px - tgt_dist
#         reverse_level = Cup
#         reverse_side  = "long"
#         d_stop_level  = ORL  # ≈ ORL (-tick if modeling ticks)

#     def first_touch(after_df, side, target, stop):
#         if side=="long":
#             hit_stop = after_df[after_df["Low"]  <= stop]
#             hit_tgt  = after_df[after_df["High"] >= target]
#         else:
#             hit_stop = after_df[after_df["High"] >= stop]
#             hit_tgt  = after_df[after_df["Low"]  <= target]
#         if not hit_stop.empty and not hit_tgt.empty:
#             t_stop, t_tgt = hit_stop.index.min(), hit_tgt.index.min()
#             return ("STOP", t_stop) if t_stop <= t_tgt else ("TARGET", t_tgt)
#         if not hit_stop.empty: return ("STOP", hit_stop.index.min())
#         if not hit_tgt.empty:  return ("TARGET", hit_tgt.index.min())
#         return (None, None)

#     trades = []
#     after = mins_day.loc[t_entry:]
#     reason1, t_exit1 = first_touch(after, side, target, stop)

#     # Optional: C reversal
#     t_rev = None
#     if allow_c_reversal:
#         touched = after[after["Low"] <= reverse_level] if side=="long" else after[after["High"] >= reverse_level]
#         if not touched.empty:
#             t_rev = touched.index.min()

#     # Choose earliest event
#     events = []
#     if t_exit1 is not None: events.append(("EXIT1", t_exit1, reason1))
#     if t_rev is not None:   events.append(("REV",   t_rev,   "C_REV"))
#     events.append(("EOD", mins_day.index[-1], "EOD"))
#     events.sort(key=lambda x: x[1])
#     event, t_evt, evt_reason = events[0]

#     if event == "EXIT1":
#         if fill_at_level and evt_reason in ("TARGET","STOP"):
#             level_px = (target if evt_reason=="TARGET" else stop)
#             exit_px = level_px - (slippage if side=="long" else -slippage)
#         else:
#             exit_px = float(mins_day.loc[t_evt, "Close"]) - (slippage if side=="long" else -slippage)
#         pnl = (exit_px - entry_px) if side=="long" else (entry_px - exit_px)
#         pnl -= cost_pts_rt
#         trades.append(dict(entry_time=t_entry, side=side, entry_px=entry_px,
#                            exit_time=t_evt, exit_px=exit_px, exit_reason=evt_reason,
#                            pnl_pts=pnl, ORH=ORH, ORL=ORL,
#                            PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
#                            A_distance=Adist, TP_mult_A=target_mult_A))
#         return trades

#     if event == "EOD":
#         exit_px = float(mins_day.loc[t_evt,"Close"]) - (slippage if side=="long" else -slippage)
#         pnl = (exit_px - entry_px) if side=="long" else (entry_px - exit_px)
#         pnl -= cost_pts_rt
#         trades.append(dict(entry_time=t_entry, side=side, entry_px=entry_px,
#                            exit_time=t_evt, exit_px=exit_px, exit_reason="EOD",
#                            pnl_pts=pnl, ORH=ORH, ORL=ORL,
#                            PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
#                            A_distance=Adist, TP_mult_A=target_mult_A))
#         return trades

#     # REV at C (single flip)
#     if fill_at_level:
#         exit_px1 = reverse_level - (slippage if side=="long" else -slippage)
#     else:
#         exit_px1 = float(mins_day.loc[t_evt,"Close"]) - (slippage if side=="long" else -slippage)
#     pnl1 = (exit_px1 - entry_px) if side=="long" else (entry_px - exit_px1)
#     pnl1 -= cost_pts_rt
#     trades.append(dict(entry_time=t_entry, side=side, entry_px=entry_px,
#                        exit_time=t_evt, exit_px=exit_px1, exit_reason="C_REV",
#                        pnl_pts=pnl1, ORH=ORH, ORL=ORL,
#                        PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
#                        A_distance=Adist, TP_mult_A=target_mult_A))

#     # Open reversed position
#     side2 = reverse_side
#     entry_px2 = exit_px1
#     after2 = mins_day.loc[t_evt:]
#     # keep TP = k × A
#     if side2=="short":
#         stop2 = d_stop_level
#         target2 = entry_px2 - tgt_dist
#     else:
#         stop2 = d_stop_level
#         target2 = entry_px2 + tgt_dist

#     reason2, t_exit2 = first_touch(after2, side2, target2, stop2)
#     if t_exit2 is None:
#         t_exit2, reason2 = after2.index[-1], "EOD"

#     if fill_at_level and reason2 in ("TARGET","STOP"):
#         level_px2 = (target2 if reason2=="TARGET" else stop2)
#         exit_px2 = level_px2 - (slippage if side2=="long" else -slippage)
#     else:
#         exit_px2 = float(mins_day.loc[t_exit2,"Close"]) - (slippage if side2=="long" else -slippage)

#     pnl2 = (exit_px2 - entry_px2) if side2=="long" else (entry_px2 - exit_px2)
#     pnl2 -= cost_pts_rt
#     trades.append(dict(entry_time=t_evt, side=side2, entry_px=entry_px2,
#                        exit_time=t_exit2, exit_px=exit_px2, exit_reason=reason2,
#                        pnl_pts=pnl2, ORH=ORH, ORL=ORL,
#                        PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
#                        A_distance=Adist, TP_mult_A=target_mult_A))
#     return trades

# def backtest_acd_pivot(meta_df, n_grid, P_grid, W=20, T=20, hold=None,
#                        slippage=0.0, cost_pts=0.0,
#                        start_date=None, end_date=None,
#                        target_mult_A=1.0, fill_at_level=True, allow_c_reversal=False):
#     """Walk-forward: choose (n,P) on prior W sessions that maximize P&L, then trade (TP = k × A)."""
#     hold = hold or (T//2)
#     all_days = meta_df.index.tolist()
#     if end_date is None: end_date = all_days[-1]
#     if start_date is None: start_date = all_days[-1] - timedelta(days=1460)
#     test_days = [d for d in all_days if (d>=start_date and d<=end_date)]

#     trades, params = [], []
#     max_n = max(n_grid)

#     for day in test_days:
#         idx = meta_df.index.get_loc(day)
#         if isinstance(idx, slice): idx = idx.start
#         if idx < (W + max_n):   # ensure enough history for both avgRange_n and WF window
#             continue
#         window_days = meta_df.index[idx-W:idx]

#         best_score, best_pair = -1e18, None
#         for n in n_grid:
#             if np.isnan(meta_df.loc[day, f"avgRange_{n}"]): continue
#             for P in P_grid:
#                 score = 0.0; ok = True
#                 for wday in window_days:
#                     r = meta_df.loc[wday]
#                     req = [r["ORH"], r["ORL"], r["PivotLow"], r["PivotHigh"], r[f"avgRange_{n}"]]
#                     if any(np.isnan(req)): ok=False; break
#                     mins = get_day_minutes(wday)
#                     if mins.empty: continue
#                     row_meta = dict(ORH=float(r["ORH"]), ORL=float(r["ORL"]),
#                                     PivotLow=float(r["PivotLow"]), PivotHigh=float(r["PivotHigh"]),
#                                     avgRange_n=float(r[f"avgRange_{n}"]))
#                     trs = simulate_day(mins, row_meta, n, P, T, hold,
#                                        slippage, cost_pts,
#                                        target_mult_A=target_mult_A,
#                                        fill_at_level=fill_at_level,
#                                        allow_c_reversal=allow_c_reversal)
#                     score += sum(t["pnl_pts"] for t in trs)
#                 if ok and score > best_score:
#                     best_score, best_pair = score, (n, P)

#         if best_pair is None:
#             continue

#         n_star, P_star = best_pair
#         mins_day = get_day_minutes(day)
#         if mins_day.empty:
#             continue
#         r = meta_df.loc[day]
#         row_meta = dict(ORH=float(r["ORH"]), ORL=float(r["ORL"]),
#                         PivotLow=float(r["PivotLow"]), PivotHigh=float(r["PivotHigh"]),
#                         avgRange_n=float(r[f"avgRange_{n_star}"]))

#         trs_d = simulate_day(mins_day, row_meta, n_star, P_star, T, hold,
#                              slippage, cost_pts,
#                              target_mult_A=target_mult_A,
#                              fill_at_level=fill_at_level,
#                              allow_c_reversal=allow_c_reversal)
#         for t in trs_d:
#             t.update({"Date": day, "n_used": n_star, "P_used": P_star, "T": T,
#                       "TP_mult_A": target_mult_A, "slippage": slippage, "cost_pts": cost_pts})
#         trades.extend(trs_d)
#         params.append({"Date": day, "n": n_star, "P": P_star, "score_window": best_score})

#     trades_df = pd.DataFrame(trades)
#     if not trades_df.empty:
#         for c in ["Date","entry_time","exit_time"]:
#             trades_df[c] = pd.to_datetime(trades_df[c])
#         trades_df = trades_df.sort_values(["Date","entry_time"]).reset_index(drop=True)
#     params_df = (pd.DataFrame(params).sort_values("Date").reset_index(drop=True)
#                  if params else pd.DataFrame(columns=["Date","n","P","score_window"]))
#     return trades_df, params_df

# # --- First tradable day respecting pre-history from meta_full ---
# first_tradable_day = meta_full.index[need_sessions]
# start_d = max(year_start.date(), first_tradable_day)
# end_d   = year_end.date()

# # --- Run WF on the full year (pass meta_full so history is intact) ---
# trades_df, params_df = backtest_acd_pivot(
#     meta_df=meta_full, n_grid=n_grid, P_grid=P_grid, W=W,
#     T=T, hold=T//2,
#     slippage=slippage_pts, cost_pts=cost_pts_rt,
#     start_date=start_d, end_date=end_d,
#     target_mult_A=target_mult_A,
#     fill_at_level=fill_at_level, allow_c_reversal=allow_c_reversal
# )

# # Keep only trades whose calendar Date falls inside the target year (safety)
# if not trades_df.empty:
#     trades_df = trades_df[(trades_df["Date"].dt.date >= year_start.date()) &
#                           (trades_df["Date"].dt.date <= year_end.date())].copy()

# # Save
# out_base = f"/home/newberry3/disha/acd_pivot_year_{YEAR}"
# os.makedirs(out_base, exist_ok=True)
# trades_path = os.path.join(out_base, f"trades_{YEAR}_k{target_mult_A}.csv")
# params_path = os.path.join(out_base, f"params_{YEAR}_k{target_mult_A}.csv")
# trades_df.to_csv(trades_path, index=False)
# params_df.to_csv(params_path, index=False)
# print(f"[OK] Buffer={buffer_days}d | Trade window {start_d}..{end_d}\nSaved:\n  {trades_path}\n  {params_path}")
# # =====================================================================================


[OK] Buffer=180d | Trade window 2021-09-24..2021-12-31
Saved:
  /home/newberry3/disha/acd_pivot_year_2021/trades_2021_k1.5.csv
  /home/newberry3/disha/acd_pivot_year_2021/params_2021_k1.5.csv


In [44]:
# ===================== ACD + Pivot (paper-faithful) rolling backtest (TP = k × A) =====================
import pandas as pd, numpy as np, os
from datetime import timedelta

# ======== USER INPUTS ========
PERIOD_START    = pd.Timestamp("2021-06-01")
PERIOD_END      = pd.Timestamp("2025-06-30")
T               = 20          # opening range minutes
W               = 20          # walk-forward window (training days)
n_grid          = list(range(12, 31, 2))            # avg-range lookbacks to scan (paper optimizes n)
P_grid          = [x/100 for x in range(10,31,2)]   # A distance = P * avgRange_n
slippage_pts    = 0.5
cost_pts_rt     = 0.5         # round-trip cost (pts)
target_mult_A   = 1           # TAKE-PROFIT = target_mult_A × A (vol-adjusted)
fill_at_level   = False       # fill exactly at target/stop when touched
allow_c_reversal= False       # allow a single C reversal after established A trade

# ======== REQUIRE: spot_1min already built (Datetime, Open, High, Low, Close), 09:15–15:25 only ========

# --- Pre-history requirement & auto buffer ---
need_sessions = W + max(n_grid)      # prior sessions required before first tradable day
buffer_days   = 25                   # good starting guess for multi-year
max_buffer    = 900

def build_meta_from_minutes(spot_slice: pd.DataFrame, T: int, n_grid: list[int]) -> pd.DataFrame:
    """Daily meta: ORH/ORL, PivotRange (PP, SV, PD => PivotLow/High), shifted rolling avgRange_n."""
    z = spot_slice.copy()
    z["session_date"] = z["Datetime"].dt.date

    # Daily OHLC
    daily = (z.groupby("session_date")
               .agg(High=("High","max"), Low=("Low","min"), Close=("Close","last"))
               .sort_index())

    # Drop incomplete days (keep ~full sessions so OR is valid)
    counts = z.groupby("session_date")["Datetime"].count()
    good_days = counts[counts >= 350].index
    daily = daily.loc[good_days]

    # Prior-day pivots
    prev = daily.shift(1).rename(columns={"High":"Prev_H","Low":"Prev_L","Close":"Prev_C"})
    D = daily.join(prev[["Prev_H","Prev_L","Prev_C"]])
    D["PP"] = (D["Prev_H"] + D["Prev_L"] + D["Prev_C"]) / 3.0
    D["SV"] = (D["Prev_H"] + D["Prev_L"]) / 2.0
    D["PD"] = D["PP"] - D["SV"]
    D["PivotLow"]  = D["PP"] - D["PD"].abs()
    D["PivotHigh"] = D["PP"] + D["PD"].abs()

    # Opening range per day (T minutes from 09:15 inclusive)
    tmp = z.copy()
    tmp["or_start"] = pd.to_datetime(tmp["session_date"].astype(str) + " 09:15:00")
    tmp["or_end"]   = tmp["or_start"] + pd.Timedelta(minutes=T-1)
    in_or = (tmp["Datetime"] >= tmp["or_start"]) & (tmp["Datetime"] <= tmp["or_end"])
    or_table = (tmp[in_or].groupby("session_date").agg(ORH=("High","max"), ORL=("Low","min")).reindex(D.index))

    # Shifted rolling daily range means (no look-ahead)
    D["DailyRange"] = D["High"] - D["Low"]
    for n in n_grid:
        D[f"avgRange_{n}"] = D["DailyRange"].rolling(n).mean().shift(1)

    meta_built = D.join(or_table)
    need_cols = ["ORH","ORL","PP","PivotLow","PivotHigh"] + [f"avgRange_{n}" for n in n_grid]
    return meta_built.dropna(subset=need_cols)

# Grow buffer until we have enough pre-history for the rolling WF
while True:
    hist_start = PERIOD_START - pd.Timedelta(days=buffer_days)
    spot_roll = spot_1min[(spot_1min["Datetime"] >= hist_start) & (spot_1min["Datetime"] <= PERIOD_END)].copy()
    meta_full = build_meta_from_minutes(spot_roll, T=T, n_grid=n_grid)
    if len(meta_full.index) > need_sessions:
        break
    buffer_days = min(buffer_days + 60, max_buffer)
    if buffer_days >= max_buffer:
        raise ValueError("Not enough pre-history even after max buffer; reduce W/max(n_grid) or extend data.")

# Prepare minute access (session_date, Datetime index) using buffered slice
minute_df = spot_roll.copy()
minute_df["session_date"] = minute_df["Datetime"].dt.date
minute_df = minute_df.set_index(["session_date","Datetime"]).sort_index()

def get_day_minutes(day):
    try:
        df = minute_df.loc[day][["Open","High","Low","Close"]].copy()
        if not isinstance(df.index, pd.DatetimeIndex):
            df = df.set_index("Datetime")
        return df
    except KeyError:
        return pd.DataFrame(columns=["Open","High","Low","Close"])

# ---- A/C levels per paper: A = P * avgRange_n ; C = 2*A (distance from OR bounds) ----
def compute_ac_levels(ORH, ORL, avg_range, P):
    A = P * float(avg_range)
    return {"A": A, "Aup": ORH + A, "Adown": ORL - A, "Cup": ORH + 2*A, "Cdown": ORL - 2*A}

# A is established when Close stays beyond threshold for ≥ T/2 minutes
def hold_established(series_close, threshold, side, hold_minutes):
    if series_close.empty: return None
    cond = (series_close >= threshold) if side=="long" else (series_close <= threshold)
    groups = (~cond).cumsum()
    run = cond.groupby(groups).cumcount() + 1
    hits = run[(cond) & (run >= hold_minutes)]
    return None if hits.empty else hits.index[0]

def simulate_day(mins_day, row_meta, n, P, T=20, hold=None,
                 slippage=0.0, cost_pts=0.0,
                 target_mult_A=1.0, fill_at_level=True, allow_c_reversal=False):
    """Paper rules with TP = target_mult_A × A.
       - A established after hold=T/2 beyond Aup/Adown → enter.
       - Stop refined by Pivot Range (long stop = max(ORL, PivotLow); short stop = min(ORH, PivotHigh)).
       - Optional single C-reversal with D stop (~ opposite OR bound).
       - Profit-taking: TP = target_mult_A × A; else EOD flat.
    """
    if mins_day.empty: return []
    hold = hold or (T//2)

    ORH, ORL = float(row_meta["ORH"]), float(row_meta["ORL"])
    Avals = compute_ac_levels(ORH, ORL, row_meta["avgRange_n"], P)
    Aup, Adown, Cup, Cdown, Adist = Avals["Aup"], Avals["Adown"], Avals["Cup"], Avals["Cdown"], Avals["A"]

    # A establishment (earliest)
    t_est_long  = hold_established(mins_day["Close"], Aup,   "long",  hold)
    t_est_short = hold_established(mins_day["Close"], Adown, "short", hold)

    cands = []
    if t_est_long  is not None:  cands.append(("long",  t_est_long))
    if t_est_short is not None:  cands.append(("short", t_est_short))
    if not cands: return []

    cands.sort(key=lambda x: x[1])
    side, t_entry = cands[0]
    entry_px = float(mins_day.loc[t_entry, "Close"]) + (slippage if side=="long" else -slippage)

    # Paper-refined stops & target = k × A
    tgt_dist = target_mult_A * Adist
    if side=="long":
        stop = max(ORL, float(row_meta["PivotLow"]))
        target = entry_px + tgt_dist
        reverse_level = Cdown
        reverse_side  = "short"
        d_stop_level  = ORH
    else:
        stop = min(ORH, float(row_meta["PivotHigh"]))
        target = entry_px - tgt_dist
        reverse_level = Cup
        reverse_side  = "long"
        d_stop_level  = ORL

    def first_touch(after_df, side, target, stop):
        if side=="long":
            hit_stop = after_df[after_df["Low"]  <= stop]
            hit_tgt  = after_df[after_df["High"] >= target]
        else:
            hit_stop = after_df[after_df["High"] >= stop]
            hit_tgt  = after_df[after_df["Low"]  <= target]
        if not hit_stop.empty and not hit_tgt.empty:
            t_stop, t_tgt = hit_stop.index.min(), hit_tgt.index.min()
            return ("STOP", t_stop) if t_stop <= t_tgt else ("TARGET", t_tgt)
        if not hit_stop.empty: return ("STOP", hit_stop.index.min())
        if not hit_tgt.empty:  return ("TARGET", hit_tgt.index.min())
        return (None, None)

    trades = []

    # ===== Monitor from the NEXT bar after entry =====
    after = mins_day.loc[mins_day.index > t_entry]   # CHANGED (was: mins_day.loc[t_entry:])
    reason1, t_exit1 = first_touch(after, side, target, stop)

    # Optional: C reversal
    t_rev = None
    if allow_c_reversal:
        touched = after[after["Low"] <= reverse_level] if side=="long" else after[after["High"] >= reverse_level]
        if not touched.empty:
            t_rev = touched.index.min()

    # Earliest event among EXIT1 / REV / EOD
    events = []
    if t_exit1 is not None: events.append(("EXIT1", t_exit1, reason1))
    if t_rev is not None:   events.append(("REV",   t_rev,   "C_REV"))
    events.append(("EOD", mins_day.index[-1], "EOD"))
    events.sort(key=lambda x: x[1])
    event, t_evt, evt_reason = events[0]

    if event == "EXIT1":
        if fill_at_level and evt_reason in ("TARGET","STOP"):
            level_px = (target if evt_reason=="TARGET" else stop)
            exit_px = level_px - (slippage if side=="long" else -slippage)
        else:
            exit_px = float(mins_day.loc[t_evt, "Close"]) - (slippage if side=="long" else -slippage)
        pnl = (exit_px - entry_px) if side=="long" else (entry_px - exit_px)
        pnl -= cost_pts_rt
        trades.append(dict(entry_time=t_entry, side=side, entry_px=entry_px,
                           exit_time=t_evt, exit_px=exit_px, exit_reason=evt_reason,
                           pnl_pts=pnl, ORH=ORH, ORL=ORL,
                           PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
                           A_distance=Adist, TP_mult_A=target_mult_A))
        return trades

    if event == "EOD":
        exit_px = float(mins_day.loc[t_evt,"Close"]) - (slippage if side=="long" else -slippage)
        pnl = (exit_px - entry_px) if side=="long" else (entry_px - exit_px)
        pnl -= cost_pts_rt
        trades.append(dict(entry_time=t_entry, side=side, entry_px=entry_px,
                           exit_time=t_evt, exit_px=exit_px, exit_reason="EOD",
                           pnl_pts=pnl, ORH=ORH, ORL=ORL,
                           PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
                           A_distance=Adist, TP_mult_A=target_mult_A))
        return trades

    # REV at C (single flip)
    if fill_at_level:
        exit_px1 = reverse_level - (slippage if side=="long" else -slippage)
    else:
        exit_px1 = float(mins_day.loc[t_evt,"Close"]) - (slippage if side=="long" else -slippage)
    pnl1 = (exit_px1 - entry_px) if side=="long" else (entry_px - exit_px1)
    pnl1 -= cost_pts_rt
    trades.append(dict(entry_time=t_entry, side=side, entry_px=entry_px,
                       exit_time=t_evt, exit_px=exit_px1, exit_reason="C_REV",
                       pnl_pts=pnl1, ORH=ORH, ORL=ORL,
                       PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
                       A_distance=Adist, TP_mult_A=target_mult_A))

    # Open reversed position
    side2 = reverse_side
    entry_px2 = exit_px1

    # ===== For the reversed leg too, monitor from the NEXT bar =====
    after2 = mins_day.loc[mins_day.index > t_evt]    # CHANGED (was: mins_day.loc[t_evt:])
    if side2=="short":
        stop2 = d_stop_level
        target2 = entry_px2 - tgt_dist
    else:
        stop2 = d_stop_level
        target2 = entry_px2 + tgt_dist

    reason2, t_exit2 = first_touch(after2, side2, target2, stop2)
    if t_exit2 is None:
        t_exit2, reason2 = mins_day.index[-1], "EOD"

    if fill_at_level and reason2 in ("TARGET","STOP"):
        level_px2 = (target2 if reason2=="TARGET" else stop2)
        exit_px2 = level_px2 - (slippage if side2=="long" else -slippage)
    else:
        exit_px2 = float(mins_day.loc[t_exit2,"Close"]) - (slippage if side2=="long" else -slippage)

    pnl2 = (exit_px2 - entry_px2) if side2=="long" else (entry_px2 - exit_px2)
    pnl2 -= cost_pts_rt
    trades.append(dict(entry_time=t_evt, side=side2, entry_px=entry_px2,
                       exit_time=t_exit2, exit_px=exit_px2, exit_reason=reason2,
                       pnl_pts=pnl2, ORH=ORH, ORL=ORL,
                       PivotLow=float(row_meta["PivotLow"]), PivotHigh=float(row_meta["PivotHigh"]),
                       A_distance=Adist, TP_mult_A=target_mult_A))
    return trades

def backtest_acd_pivot(meta_df, n_grid, P_grid, W=20, T=20, hold=None,
                       slippage=0.0, cost_pts=0.0,
                       start_date=None, end_date=None,
                       target_mult_A=1.0, fill_at_level=True, allow_c_reversal=False):
    """Walk-forward: choose (n,P) on prior W sessions that maximize P&L, then trade (TP = k × A)."""
    hold = hold or (T//2)
    all_days = meta_df.index.tolist()
    if end_date is None: end_date = all_days[-1]
    if start_date is None: start_date = all_days[-1] - timedelta(days=1460)
    test_days = [d for d in all_days if (d>=start_date and d<=end_date)]

    trades, params = [], []
    max_n = max(n_grid)

    for day in test_days:
        idx = meta_df.index.get_loc(day)
        if isinstance(idx, slice): idx = idx.start
        if idx < (W + max_n):   # ensure enough history for both avgRange_n and WF window
            continue
        window_days = meta_df.index[idx-W:idx]

        best_score, best_pair = -1e18, None
        for n in n_grid:
            if np.isnan(meta_df.loc[day, f"avgRange_{n}"]): continue
            for P in P_grid:
                score = 0.0; ok = True
                for wday in window_days:
                    r = meta_df.loc[wday]
                    req = [r["ORH"], r["ORL"], r["PivotLow"], r["PivotHigh"], r[f"avgRange_{n}"]]
                    if any(np.isnan(req)): ok=False; break
                    mins = get_day_minutes(wday)
                    if mins.empty: continue
                    row_meta = dict(ORH=float(r["ORH"]), ORL=float(r["ORL"]),
                                    PivotLow=float(r["PivotLow"]), PivotHigh=float(r["PivotHigh"]),
                                    avgRange_n=float(r[f"avgRange_{n}"]))
                    trs = simulate_day(mins, row_meta, n, P, T, hold,
                                       slippage, cost_pts,
                                       target_mult_A=target_mult_A,
                                       fill_at_level=fill_at_level,
                                       allow_c_reversal=allow_c_reversal)
                    score += sum(t["pnl_pts"] for t in trs)
                if ok and score > best_score:
                    best_score, best_pair = score, (n, P)

        if best_pair is None:
            continue

        n_star, P_star = best_pair
        mins_day = get_day_minutes(day)
        if mins_day.empty:
            continue
        r = meta_df.loc[day]
        row_meta = dict(ORH=float(r["ORH"]), ORL=float(r["ORL"]),
                        PivotLow=float(r["PivotLow"]), PivotHigh=float(r["PivotHigh"]),
                        avgRange_n=float(r[f"avgRange_{n_star}"]))

        trs_d = simulate_day(mins_day, row_meta, n_star, P_star, T, hold,
                             slippage, cost_pts,
                             target_mult_A=target_mult_A,
                             fill_at_level=fill_at_level,
                             allow_c_reversal=allow_c_reversal)
        for t in trs_d:
            t.update({"Date": day, "n_used": n_star, "P_used": P_star, "T": T,
                      "TP_mult_A": target_mult_A, "slippage": slippage, "cost_pts": cost_pts})
        trades.extend(trs_d)
        params.append({"Date": day, "n": n_star, "P": P_star, "score_window": best_score})

    trades_df = pd.DataFrame(trades)
    if not trades_df.empty:
        for c in ["Date","entry_time","exit_time"]:
            trades_df[c] = pd.to_datetime(trades_df[c])
        trades_df = trades_df.sort_values(["Date","entry_time"]).reset_index(drop=True)
    params_df = (pd.DataFrame(params).sort_values("Date").reset_index(drop=True)
                 if params else pd.DataFrame(columns=["Date","n","P","score_window"]))
    return trades_df, params_df

# --- Compute first tradable day respecting pre-history from meta_full ---
first_tradable_day = meta_full.index[need_sessions]
start_d = max(PERIOD_START.date(), first_tradable_day)
end_d   = PERIOD_END.date()
print(f"[Info] Buffer used: {buffer_days}d | need_sessions={need_sessions} | "
      f"first_tradable_day={first_tradable_day} | trade window {start_d}..{end_d}")

# --- Run WF across the entire rolling window (meta_full keeps history intact) ---
trades_df, params_df = backtest_acd_pivot(
    meta_df=meta_full, n_grid=n_grid, P_grid=P_grid, W=W,
    T=T, hold=T//2,
    slippage=slippage_pts, cost_pts=cost_pts_rt,
    start_date=start_d, end_date=end_d,
    target_mult_A=target_mult_A,
    fill_at_level=fill_at_level, allow_c_reversal=allow_c_reversal
)

# Keep only trades whose calendar Date falls inside the requested window (safety)
if not trades_df.empty:
    trades_df = trades_df[(trades_df["Date"].dt.date >= PERIOD_START.date()) &
                          (trades_df["Date"].dt.date <= PERIOD_END.date())].copy()

# --- Save ---
out_base = f"/home/newberry3/disha/acd_pivot_roll_202106_202506"
os.makedirs(out_base, exist_ok=True)
tag = f"k{target_mult_A}"
trades_path = os.path.join(out_base, f"trades_202106_202506_{tag}.csv")
params_path = os.path.join(out_base, f"params_202106_202506_{tag}.csv")
trades_df.to_csv(trades_path, index=False)
params_df.to_csv(params_path, index=False)
print(f"[OK] Saved:\n  {trades_path}\n  {params_path}")
# =====================================================================================


[Info] Buffer used: 25d | need_sessions=50 | first_tradable_day=2021-09-24 | trade window 2021-09-24..2025-06-30
[OK] Saved:
  /home/newberry3/disha/acd_pivot_roll_202106_202506/trades_202106_202506_k1.csv
  /home/newberry3/disha/acd_pivot_roll_202106_202506/params_202106_202506_k1.csv


In [46]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==================== CONFIG ====================
csv_path = Path("/home/newberry3/disha/acd_pivot_roll_202106_202506/trades_202106_202506_k1.csv")
out_xlsx = csv_path.parent / "signals_analytics_report_k1.xlsx"

# ==================== LOAD & PREP ====================
df = pd.read_csv(csv_path, parse_dates=["Date","entry_time","exit_time"])
if df.empty:
    raise ValueError("No trades found in the CSV.")

# Normalize column names to match your example style where needed
trades_df = df.copy()

# Map side to Bullish/Bearish labels expected by your summary example
side_map = {"long": "Bullish", "short": "Bearish"}
trades_df["Side"] = trades_df["side"].map(side_map).fillna(trades_df.get("side", ""))
# Map PnL columns expected by your example
# If you only have 'pnl_pts', use it for both Net and Gross (adjust if you track costs separately)
trades_df["Net_PnL_With_Spread"] = trades_df["pnl_pts"]
trades_df["Gross_PnL"] = trades_df["pnl_pts"]

# Align time columns to example names
trades_df["Entry Time"] = pd.to_datetime(trades_df["entry_time"])
trades_df["Exit Time"]  = pd.to_datetime(trades_df["exit_time"])
trades_df["Year"] = trades_df["Exit Time"].dt.year
trades_df["Date_only"] = trades_df["Exit Time"].dt.date  # keep original Date too if you want
trades_df = trades_df.sort_values("Exit Time").reset_index(drop=True)

# ============ HELPERS ============
def calculate_drawdown(pnl_series: pd.Series):
    """pnl_series is a sequence of (daily) PnL numbers."""
    cumulative = pnl_series.cumsum()
    high_watermark = cumulative.cummax()
    drawdown = cumulative - high_watermark
    max_dd = float(drawdown.min()) if not drawdown.empty else 0.0
    max_dd_pct = (max_dd / float(high_watermark.max())) if high_watermark.max() not in (0, np.nan) else 0.0
    return max_dd, max_dd_pct

def sortino_ratio(returns: pd.Series, periods: int):
    """returns = daily returns (not cumulative). 'periods' is the annualization factor (e.g., 252 for days)."""
    returns = pd.Series(returns).dropna()
    if returns.empty:
        return np.nan
    mean_return = returns.mean()
    downside = returns[returns < 0]
    downside_deviation = downside.std(ddof=1) if len(downside) > 0 else np.nan
    if pd.isna(downside_deviation) or downside_deviation == 0:
        return np.nan
    return float(mean_return / downside_deviation * np.sqrt(periods))

# ============ PERIOD DEFINITIONS ============
last_exit = trades_df["Exit Time"].max()
years = sorted(trades_df["Year"].dropna().unique().tolist())

# [start, end) intervals
periods = {f"Year {y}": [pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y+1}-01-01")] for y in years}
periods.update({
    "Last 6 Months": [last_exit - pd.DateOffset(months=6), last_exit],
    "Last 1 Year":   [last_exit - pd.DateOffset(years=1),   last_exit],
    "Last 2 Years":  [last_exit - pd.DateOffset(years=2),   last_exit],
    "Full Period":   [trades_df["Exit Time"].min(),         trades_df["Exit Time"].max()]
})

# ============ BUILD SUMMARY ============
summary_rows = []

for period_name, (start_t, end_t) in periods.items():
    chunk = trades_df[(trades_df["Exit Time"] >= start_t) & (trades_df["Exit Time"] < end_t)].copy()
    if chunk.empty:
        continue

    # Daily aggregation for return/Drawdown/Sortino
    chunk["Date"] = chunk["Exit Time"].dt.date
    daily_df = (chunk.groupby("Date")
                     .agg(Net_PnL_With_Spread=("Net_PnL_With_Spread","sum"),
                          Gross_PnL=("Gross_PnL","sum"))
                     .reset_index())

    # "Daily_Return" proxy as in your example (normalized by total abs PnL in period)
    denom = daily_df["Net_PnL_With_Spread"].abs().sum()
    if denom == 0:
        daily_df["Daily_Return"] = 0.0
    else:
        daily_df["Daily_Return"] = daily_df["Net_PnL_With_Spread"] / denom

    returns = daily_df["Daily_Return"].fillna(0.0)

    # Overall stats
    win_rate = float((chunk["Net_PnL_With_Spread"] > 0).mean()) if len(chunk) else np.nan
    max_dd, max_dd_pct = calculate_drawdown(daily_df["Net_PnL_With_Spread"])
    # Use 252 as trading days for daily Sortino annualization
    sortino_252 = sortino_ratio(returns, 252)
    total_return = float(daily_df["Net_PnL_With_Spread"].sum())
    # Annualized return from mean daily return
    mean_daily_ret = float(returns.mean()) if len(returns) else np.nan
    annualized_return = (1 + mean_daily_ret) ** 252 - 1 if not np.isnan(mean_daily_ret) else np.nan

    # Win/Loss distribution
    wins = chunk.loc[chunk["Net_PnL_With_Spread"] > 0, "Net_PnL_With_Spread"]
    losses = chunk.loc[chunk["Net_PnL_With_Spread"] < 0, "Net_PnL_With_Spread"]
    std_win = float(wins.std(ddof=1)) if not wins.empty else np.nan
    std_loss = float(losses.std(ddof=1)) if not losses.empty else np.nan
    avg_win = float(wins.mean()) if not wins.empty else np.nan
    avg_loss = float(losses.mean()) if not losses.empty else np.nan

    # Bullish/Bearish splits
    bull = chunk[chunk["Side"] == "Bullish"]
    bear = chunk[chunk["Side"] == "Bearish"]

    def side_stats(sdf: pd.DataFrame):
        pnl = float(sdf["Net_PnL_With_Spread"].sum()) if not sdf.empty else 0.0
        winp = float((sdf["Net_PnL_With_Spread"] > 0).mean()*100) if not sdf.empty else np.nan
        if sdf.empty:
            dd_val, dd_pct = 0.0, 0.0
        else:
            _d = (sdf.assign(Date=sdf["Exit Time"].dt.date)
                    .groupby("Date")["Net_PnL_With_Spread"].sum())
            dd_val, dd_pct = calculate_drawdown(_d)
            dd_pct *= 100
        return pnl, winp, dd_val, dd_pct

    bullish_pnl, bullish_win, bull_dd, bull_dd_pct = side_stats(bull)
    bearish_pnl, bearish_win, bear_dd, bear_dd_pct = side_stats(bear)

    summary_rows.append({
        "Period": period_name,
        "Total Net PnL (With Spread)": total_return,
        "Total Gross PnL": float(daily_df["Gross_PnL"].sum()),
        "Bullish PnL": bullish_pnl,
        "Bullish Win %": round(bullish_win, 2) if pd.notnull(bullish_win) else np.nan,
        "Bullish Max DD": bull_dd,
        "Bullish Max DD %": bull_dd_pct,
        "Bearish PnL": bearish_pnl,
        "Bearish Win %": round(bearish_win, 2) if pd.notnull(bearish_win) else np.nan,
        "Bearish Max DD": bear_dd,
        "Bearish Max DD %": bear_dd_pct,
        "Win Rate %": round(win_rate * 100, 2) if not np.isnan(win_rate) else np.nan,
        "Sortino (daily, 252)": sortino_252,
        "Max Drawdown": max_dd,
        "Max Drawdown %": max_dd_pct * 100,
        "Annualized Return %": round(annualized_return * 100, 2) if not np.isnan(annualized_return) else np.nan,
        "Mean Daily Return": mean_daily_ret,
        "Std Dev Daily Return": float(returns.std(ddof=1)) if len(returns) else np.nan,
        "Std Win": std_win,
        "Std Loss": std_loss,
        "Avg Win": avg_win,
        "Avg Loss": avg_loss
    })

summary_df = pd.DataFrame(summary_rows)

# ============ WRITE EXCEL (2 sheets) ============
with pd.ExcelWriter(out_xlsx, engine="xlsxwriter",
                    datetime_format="yyyy-mm-dd hh:mm",
                    date_format="yyyy-mm-dd") as writer:
    # Sheet 1: full trades (keep all original cols plus our added)
    trades_cols = [
        "Date","Date_only","Entry Time","Exit Time","Side","side","exit_reason",
        "entry_px","exit_px","pnl_pts","Net_PnL_With_Spread","Gross_PnL",
        "ORH","ORL","PivotLow","PivotHigh","A_distance",
        "n_used","P_used","T","slippage","cost_pts"
    ]
    trades_cols = [c for c in trades_cols if c in trades_df.columns]
    trades_df[trades_cols].to_excel(writer, sheet_name="All_Trades", index=False)

    # Sheet 2: analytics summary
    summary_df.to_excel(writer, sheet_name="Analytics_Summary", index=False)

    # Optional autosize
    wb = writer.book
    for sheet_name, src in [("All_Trades", trades_df[trades_cols]), ("Analytics_Summary", summary_df)]:
        ws = writer.sheets[sheet_name]
        for i, col in enumerate(src.columns):
            max_len = max(len(str(col)), *(len(str(v)) for v in src[col].astype(str).head(1000)))
            ws.set_column(i, i, min(max_len + 2, 50))

print(f"✅ Excel written: {out_xlsx}")


✅ Excel written: /home/newberry3/disha/acd_pivot_roll_202106_202506/signals_analytics_report_k1.xlsx


In [7]:
#LOAD AND SAVE OPTION DATA SEPARATELY
years_to_load = [2021, 2022, 2023, 2024, 2025]

outdir = "./options_data_yearwise"
os.makedirs(outdir, exist_ok=True)

for year in years_to_load:
    print(f"\n--- Loading option data for year: {year} ---")

    # === NEAREST EXPIRY ===
    options_files = glob.glob(f"/home/newberry3/main/Data/NIFTY/NIFTY_{year}*.pkl")
    if not options_files:
        print(f"No .pkl files found in /main/Data/NIFTY for year {year}")
        continue

    all_cols = set()
    dfs = []
    for file in options_files:
        df = pickle.load(open(file, "rb"))
        df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
        all_cols.update(df.columns)
    all_cols = sorted(list(all_cols))

    for file in options_files:
        df = pickle.load(open(file, "rb"))
        df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
        for col in all_cols:
            if col not in df.columns:
                df[col] = pd.NA
        df = df[all_cols]
        dfs.append(df)
    options_df = pd.concat(dfs, ignore_index=True)
    for col in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
        if col in options_df.columns:
            options_df[col] = options_df[col].astype(str)
    options_df['DateTime'] = pd.to_datetime(options_df['Date'] + " " + options_df['Time'], errors='coerce')
    options_df = options_df.dropna(subset=['DateTime'])
    cols_to_drop = ['Ticker', 'High', 'Low', 'Close', 'Date','Time']
    options_df = options_df.drop(columns=[col for col in cols_to_drop if col in options_df.columns])
    cols = ['DateTime'] + [col for col in options_df.columns if col != 'DateTime']
    options_df = options_df[cols]
    options_df['StrikePrice'] = options_df['StrikePrice'].astype(float)
    options_df['Type'] = options_df['Type'].str.strip().str.upper()
    options_df['ExpiryDate'] = pd.to_datetime(options_df['ExpiryDate']).dt.normalize()

    # --- Save nearest expiry for this year ---
    nearest_path = os.path.join(outdir, f"options_nearest_{year}.parquet")
    options_df.to_parquet(nearest_path, index=False)
    print(f"✅ Saved NEAREST expiry options for {year} to {nearest_path}")

    # === NEXT EXPIRY ===
    options_files_2 = glob.glob(f"/home/newberry3/main/Data/2ndweeknext_Expiry/NIFTY_{year}*.pkl")
    if not options_files_2:
        print(f"No .pkl files found in /2ndweeknext_Expiry for year {year}")
        continue

    all_cols_2 = set()
    for file in options_files_2:
        df_2 = pickle.load(open(file, "rb"))
        df_2 = df_2.drop(columns=[col for col in ['OI', 'Volume'] if col in df_2.columns], errors='ignore')
        all_cols_2.update(df_2.columns)
    all_cols_2 = sorted(list(all_cols_2))

    # dfs_2 = []
    # for file in options_files_2:
    #     df_2 = pickle.load(open(file, "rb"))
    #     df_2 = df_2.drop(columns=[col for col in ['OI', 'Volume'] if col in df_2.columns], errors='ignore')
    #     for col in all_cols_2:
    #         if col not in df_2.columns:
    #             df_2[col] = pd.NA
    #     df_2 = df_2[all_cols_2]
    #     # Drop NA in key columns
    #     key_cols = ['StrikePrice', 'ExpiryDate', 'Date', 'Time', 'Type']
    #     df_2 = df_2.dropna(subset=[col for col in key_cols if col in df_2.columns])
    #     # Parse DateTime
    #     dt_strings = df_2['Date'].astype(str) + ' ' + df_2['Time'].astype(str)
    #     df_2['DateTime'] = pd.to_datetime(dt_strings, errors='coerce')
    #     df_2 = df_2.dropna(subset=['DateTime'])
    #     dfs_2.append(df_2)
    # options_df_2 = pd.concat(dfs_2, ignore_index=True)
    # for col in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
    #     if col in options_df_2.columns:
    #         options_df_2[col] = options_df_2[col].astype(str)
    # options_df_2['StrikePrice'] = options_df_2['StrikePrice'].astype(float)
    # options_df_2['Type'] = options_df_2['Type'].str.strip().str.upper()
    # options_df_2['ExpiryDate'] = pd.to_datetime(options_df_2['ExpiryDate']).dt.normalize()
    # cols_to_drop = ['Ticker', 'High', 'Low', 'Close', 'Date','Time']
    # options_df_2 = options_df_2.drop(columns=[col for col in cols_to_drop if col in options_df_2.columns])
    # cols = ['DateTime'] + [col for col in options_df_2.columns if col != 'DateTime']
    # options_df_2 = options_df_2[cols]

    # # --- Save next expiry for this year ---
    # next_path = os.path.join(outdir, f"options_next_{year}.parquet")
    # options_df_2.to_parquet(next_path, index=False)
    # print(f"✅ Saved NEXT expiry options for {year} to {next_path}")

print(f"\nAll options data saved to {outdir}")



--- Loading option data for year: 2021 ---


/tmp/ipykernel_509678/2332700076.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  options_df = pd.concat(dfs, ignore_index=True)


✅ Saved NEAREST expiry options for 2021 to ./options_data_yearwise/options_nearest_2021.parquet

--- Loading option data for year: 2022 ---
✅ Saved NEAREST expiry options for 2022 to ./options_data_yearwise/options_nearest_2022.parquet

--- Loading option data for year: 2023 ---
✅ Saved NEAREST expiry options for 2023 to ./options_data_yearwise/options_nearest_2023.parquet

--- Loading option data for year: 2024 ---
✅ Saved NEAREST expiry options for 2024 to ./options_data_yearwise/options_nearest_2024.parquet

--- Loading option data for year: 2025 ---
✅ Saved NEAREST expiry options for 2025 to ./options_data_yearwise/options_nearest_2025.parquet

All options data saved to ./options_data_yearwise


In [9]:
#REQUIRED FOR ALL
data_dir = "./options_data_yearwise"

# Dictionaries to store DataFrames by year
nearest_dict = {}
# next_dict = {}
merged_dict = {}

for year in years_to_load:
    nearest_path = os.path.join(data_dir, f"options_nearest_{year}.parquet")
    # next_path = os.path.join(data_dir, f"options_next_{year}.parquet")

    # Load each DataFrame and assign to dynamic variable names
    try:
        options_df = pd.read_parquet(nearest_path)
        # options_df_2 = pd.read_parquet(next_path)
        # print(f"Loaded: {nearest_path} and {next_path}")
    except Exception as e:
        print(f"Skipping year {year} due to load error: {e}")
        continue

    # Store individually
    nearest_dict[year] = options_df
    # next_dict[year] = options_df_2

    # Merge as per your logic
    merged = pd.concat([options_df], ignore_index=True)
    merged = merged.drop_duplicates(
        subset=['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'], keep='first'
    ).reset_index(drop=True)
    merged = merged.sort_values(['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'])

    # Store merged DataFrame
    merged_dict[year] = merged

    # Optionally: Save merged for each year
    merged_path = os.path.join(data_dir, f"options_df_merged_{year}.parquet")
    merged.to_parquet(merged_path, index=False)
    print(f"✅ Saved merged DataFrame for {year} to {merged_path}")

# Now you have:
# nearest_dict[2024], next_dict[2024], merged_dict[2024], etc.
# Or if you want variables, you can also:
for year in years_to_load:
    globals()[f"options_df_{year}"] = nearest_dict.get(year)
    # globals()[f"options_df_2_{year}"] = next_dict.get(year)
    globals()[f"options_df_merged_{year}"] = merged_dict.get(year)

✅ Saved merged DataFrame for 2021 to ./options_data_yearwise/options_df_merged_2021.parquet
✅ Saved merged DataFrame for 2022 to ./options_data_yearwise/options_df_merged_2022.parquet
✅ Saved merged DataFrame for 2023 to ./options_data_yearwise/options_df_merged_2023.parquet
✅ Saved merged DataFrame for 2024 to ./options_data_yearwise/options_df_merged_2024.parquet
✅ Saved merged DataFrame for 2025 to ./options_data_yearwise/options_df_merged_2025.parquet


In [10]:
# BS/IV functions

# --- Black-Scholes Option Price ---
def black_scholes_price(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return 0

    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        return None
    
# --- Implied Volatility from Option Price ---
def implied_volatility(option_price, S, K, T, r, option_type):
    """
    Uses Brent's method to find implied volatility from the market price.
    """
    try:
        return brentq(
            lambda sigma: black_scholes_price(S, K, T, r, sigma, option_type) - option_price,
            a=0.01,
            b=3.0,
            maxiter=1000,
            xtol=1e-6
        )
    except (ValueError, RuntimeError):
        return None
    
# --- Black-Scholes Greeks ---
def black_scholes_greeks(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return None

    # Use synthetic future price
    F = S * np.exp(r * T)

    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        delta = np.exp(-r * T) * norm.cdf(d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        delta = -np.exp(-r * T) * norm.cdf(-d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    gamma = norm.pdf(d1) / (F * sigma * np.sqrt(T))
    vega = F * norm.pdf(d1) * np.sqrt(T) / 100

    return {
        'Delta': round(delta, 5),
        'Gamma': round(gamma, 5),
        'Vega': round(vega, 5),
        'Theta': round(theta, 5),
        'Rho': round(rho, 5)
    }

# --- Time to Expiry (Fractional) ---
def calculate_time_to_expiry(manual_datetime_str, expiry_date_str):
    now = datetime.strptime(manual_datetime_str, "%Y-%m-%d %H:%M:%S")
    expiry_date = datetime.strptime(expiry_date_str, "%d-%m-%y").date()

    market_open = time(9, 15)
    market_close = time(15, 30)
    today = now.date()
    days_left = (expiry_date - today).days

    if days_left <= 0:
        days_left += 1

    total_trading_minutes = (market_close.hour * 60 + market_close.minute) - (market_open.hour * 60 + market_open.minute)
    current_minutes_since_open = (now.hour * 60 + now.minute) - (market_open.hour * 60 + market_open.minute)

    if current_minutes_since_open < 0:
        T = round(days_left / 365, 6)
    elif current_minutes_since_open >= total_trading_minutes:
        T = round(max(0, (days_left - 1) / 365), 6)
    else:
        fraction_of_day_passed = current_minutes_since_open / total_trading_minutes
        T = round((days_left - fraction_of_day_passed) / 365, 6)

    print(f"\nManual Time Entered: {now}")
    print(f"Expiry Date: {expiry_date}")
    print(f"Days to Expiry (with fraction): {days_left - (fraction_of_day_passed if 0 <= current_minutes_since_open < total_trading_minutes else 0):.6f}")
    print(f"Time to Expiry in Years (T): {T}")

    return T

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---- Load ACD trades in the given format ----
trades_path = Path("/home/newberry3/disha/acd_pivot_roll_202106_202506/trades_202106_202506_k1.0.csv")
trades_df = pd.read_csv(trades_path, parse_dates=["entry_time","exit_time","Date"])

# Map to the fields expected by the strike selector:
#   - 'Entry Time', 'Entry Price', 'Side' ('Bullish'/'Bearish')
trade_log_df = trades_df.rename(columns={
    "entry_time": "Entry Time",
    "entry_px":   "Entry Price",
    "exit_time":  "Exit Time",
    "side":       "SideRaw"
})
trade_log_df["Side"] = trade_log_df["SideRaw"].str.lower().map({"long":"Bullish","short":"Bearish"})
trade_log_df.drop(columns=["SideRaw"], inplace=True)
trade_log_df["Year"] = trade_log_df["Entry Time"].dt.year

# ---- Strike selection without Friday/next-expiry logic ----
def find_best_strike_for_signal(row, options_df, r=0.066):
    """
    Pick the best strike (closest to target delta) for the signal's entry timestamp
    using a single options_df. Earliest expiry >= entry_time is chosen.
    Assumes the existence of: calculate_time_to_expiry, implied_volatility, black_scholes_greeks
    """
    entry_time   = pd.to_datetime(row['Entry Time'])
    spot         = float(row['Entry Price'])
    side         = row['Side']                     # 'Bullish' or 'Bearish'
    opt_type     = 'PE' if side == 'Bullish' else 'CE'   # keep same mapping you used earlier
    target_delta = -0.4 if side == 'Bullish' else 0.4

    df = options_df.copy()
    df['StrikePrice'] = df['StrikePrice'].astype(float)
    df['ExpiryDate']  = pd.to_datetime(df['ExpiryDate'])
    df['DateTime']    = pd.to_datetime(df['DateTime'])
    df['Type']        = df['Type'].astype(str)

    # earliest expiry on/after entry
    expiry_list = df.loc[df['DateTime'] >= entry_time, 'ExpiryDate'].drop_duplicates().sort_values()
    if expiry_list.empty:
        print(f"[DEBUG] No expiry ≥ entry_time for {entry_time} ({side})")
        return None
    expiry = pd.to_datetime(expiry_list.iloc[0])

    # filter to rows at entry minute (+/- 1 min), chosen expiry, and type
    time_window = pd.Timedelta(minutes=1)
    time_diff   = (df['DateTime'] - entry_time).abs()
    mask = (
        (df['DateTime'].dt.floor('min') == entry_time.floor('min')) &
        (df['ExpiryDate'] == expiry) &
        (df['Type'] == opt_type) &
        (time_diff <= time_window)
    )
    df_opts = df[mask].copy()

    print(f"[DEBUG] Entry: {entry_time}, Expiry: {expiry.date()}, Type: {opt_type}, Rows: {len(df_opts)}")
    if df_opts.empty:
        return None

    best_row = None
    best_delta_diff = np.inf

    for _, opt in df_opts.iterrows():
        K            = float(opt['StrikePrice'])
        option_price = float(opt['Open'])
        T = calculate_time_to_expiry(entry_time.strftime("%Y-%m-%d %H:%M:%S"),
                                     expiry.strftime("%d-%m-%y"))
        option_type_bs = 'put' if opt_type == 'PE' else 'call'

        iv = implied_volatility(option_price, spot, K, T, r, option_type_bs)
        if iv is None or iv <= 0:
            continue

        greeks = black_scholes_greeks(spot, K, T, r, iv, option_type_bs)
        if greeks is None:
            continue

        delta = float(greeks['Delta'])
        delta_diff = abs(delta - target_delta)

        if delta_diff < best_delta_diff:
            best_delta_diff = delta_diff
            best_row = {
                'Entry Time': entry_time,
                'Side': side,
                'Strike': K,
                'Expiry': expiry,
                'IV': iv,
                'Delta': delta,
                'Option Type': opt_type,
                'Option Price': option_price
            }

    return best_row

# ---- Run year-wise with one options_df per year in your environment (options_df_YYYY) ----
all_results = []
for year in sorted(trade_log_df["Year"].unique()):
    print(f"Processing {year}...")
    trades_y = trade_log_df[trade_log_df["Year"] == year]
    options_df = globals().get(f"options_df_{year}")
    if options_df is None:
        print(f"  [WARN] Missing options_df_{year}; skipping this year.")
        continue

    for _, row in trades_y.iterrows():
        res = find_best_strike_for_signal(row, options_df)
        if res is not None:
            all_results.append(res)

strikes_df = pd.DataFrame(all_results)
print(strikes_df)


Processing 2021...
[DEBUG] Entry: 2021-09-24 13:35:00, Expiry: 2021-09-30, Type: CE, Rows: 37

Manual Time Entered: 2021-09-24 13:35:00
Expiry Date: 2021-09-30
Days to Expiry (with fraction): 5.306667
Time to Expiry in Years (T): 0.014539

Manual Time Entered: 2021-09-24 13:35:00
Expiry Date: 2021-09-30
Days to Expiry (with fraction): 5.306667
Time to Expiry in Years (T): 0.014539

Manual Time Entered: 2021-09-24 13:35:00
Expiry Date: 2021-09-30
Days to Expiry (with fraction): 5.306667
Time to Expiry in Years (T): 0.014539

Manual Time Entered: 2021-09-24 13:35:00
Expiry Date: 2021-09-30
Days to Expiry (with fraction): 5.306667
Time to Expiry in Years (T): 0.014539

Manual Time Entered: 2021-09-24 13:35:00
Expiry Date: 2021-09-30
Days to Expiry (with fraction): 5.306667
Time to Expiry in Years (T): 0.014539

Manual Time Entered: 2021-09-24 13:35:00
Expiry Date: 2021-09-30
Days to Expiry (with fraction): 5.306667
Time to Expiry in Years (T): 0.014539

Manual Time Entered: 2021-09-24 13:

In [12]:
import pandas as pd
from pathlib import Path

# ---------- 1) Load ACD trades (rolling CSV) ----------
trades_path = Path("/home/newberry3/disha/acd_pivot_roll_202106_202506/trades_202106_202506_k1.0.csv")
trades_df = pd.read_csv(
    trades_path,
    parse_dates=["entry_time","exit_time","Date"]
)

# Ensure expected columns exist (light guard)
need_cols = {
    "entry_time","exit_time","entry_px","exit_px","side","exit_reason","pnl_pts",
    "ORH","ORL","PivotLow","PivotHigh","A_distance","TP_mult_A","Date",
    "n_used","P_used","T","slippage","cost_pts"
}
missing = need_cols - set(trades_df.columns)
if missing:
    raise ValueError(f"Trades CSV is missing required columns: {missing}")

# Normalize & create the trade_log_df used in merge
trade_log_df = trades_df.rename(columns={
    "entry_time": "Entry Time",
    "exit_time":  "Exit Time",
    "entry_px":   "Entry Price",
    "exit_px":    "Exit Price",
    "side":       "Side"
}).copy()

# Optional: map 'Side' to Bullish/Bearish if you used that convention elsewhere
# trade_log_df["SideMapped"] = trade_log_df["Side"].str.lower().map({"long":"Bullish","short":"Bearish"})

# ---------- 2) Assume strikes_df already exists from your strike selector ----------
# It should contain at least: ['Entry Time','Strike','Expiry','Option Type','Option Price','IV','Delta', 'Side']
# If your object is 'strikes_df_sorted', normalize it here:
try:
    strikes_df_base = strikes_df.copy()
except NameError:
    strikes_df_base = strikes_df.copy()

# Ensure datetime type and a stable order column so we can preserve strikes order after merge
strikes_df_base["Entry Time"] = pd.to_datetime(strikes_df_base["Entry Time"])
strikes_df_base["_order"] = range(len(strikes_df_base))

# ---------- 3) Merge on Entry Time (preserving strikes order) ----------
# Choose the important trade columns you want carried into the merged view
trade_cols_to_attach = [
    "Entry Time","Exit Time","Date","Side","Entry Price","Exit Price","exit_reason","pnl_pts",
    "ORH","ORL","PivotLow","PivotHigh","A_distance","TP_mult_A","n_used","P_used","T","slippage","cost_pts"
]
trade_cols_to_attach = [c for c in trade_cols_to_attach if c in trade_log_df.columns]

strikes_df_merged = (
    strikes_df_base
      .merge(trade_log_df[trade_cols_to_attach], on="Entry Time", how="left", suffixes=("","_trade"))
      .sort_values("_order")
      .drop(columns=["_order"])
      .reset_index(drop=True)
)

# ---------- 4) Optional: column ordering for readability ----------
opt_cols = ["Entry Time","Exit Time","Date","Side",             # timeline + direction
            "Entry Price","Exit Price","pnl_pts","exit_reason", # PnL cluster
            "Strike","Option Type","Expiry","Option Price","IV","Delta",  # option chosen
            "ORH","ORL","PivotLow","PivotHigh","A_distance","TP_mult_A",  # day context
            "n_used","P_used","T","slippage","cost_pts"]                 # config
# Keep only those that exist
ordered_cols = [c for c in opt_cols if c in strikes_df_merged.columns]
# Include any other columns that might be present but not listed
remaining = [c for c in strikes_df_merged.columns if c not in ordered_cols]
strikes_df_merged = strikes_df_merged[ordered_cols + remaining]

print(strikes_df_merged.head())
# strikes_df_merged now has both the option selection and the full trade context


           Entry Time           Exit Time       Date     Side  Entry Price  \
0 2021-09-24 13:35:00 2021-09-24 13:35:00 2021-09-24  Bearish     17846.00   
1 2021-09-27 10:48:00 2021-09-27 12:55:00 2021-09-27  Bearish     17836.75   
2 2021-09-28 11:19:00 2021-09-28 11:52:00 2021-09-28  Bearish     17797.10   
3 2021-09-29 13:19:00 2021-09-29 13:19:00 2021-09-29  Bullish     17697.65   
4 2021-09-30 13:25:00 2021-09-30 14:26:00 2021-09-30  Bearish     17626.05   

     Exit Price    pnl_pts exit_reason   Strike Option Type  ...  \
0  17797.341667  48.158333        STOP  17950.0          CE  ...   
1  17884.025000 -47.775000        STOP  17900.0          CE  ...   
2  17782.699167  13.900833      TARGET  17850.0          CE  ...   
3  17743.891667  45.741667        STOP  17650.0          PE  ...   
4  17610.587500  14.962500      TARGET  17650.0          CE  ...   

       PivotLow     PivotHigh  A_distance  TP_mult_A  n_used  P_used   T  \
0  17745.225000  17796.841667   42.232750     

In [13]:
# Combine all yearwise merged DataFrames into one
options_df_merged_all = pd.concat(merged_dict.values(), ignore_index=True)

# (Optional) Drop duplicates across all years just in case
options_df_merged_all = options_df_merged_all.drop_duplicates(
    subset=['DateTime', 'ExpiryDate', 'StrikePrice', 'Type'], keep='first'
).reset_index(drop=True)

# (Optional) Sort for easier use
options_df_merged_all = options_df_merged_all.sort_values(
    ['DateTime', 'ExpiryDate', 'StrikePrice', 'Type']
).reset_index(drop=True)

# Save if needed
options_df_merged_all_path = os.path.join(data_dir, "options_df_merged_all.parquet")
options_df_merged_all.to_parquet(options_df_merged_all_path, index=False)
print(f"✅ Saved ALL merged options data to {options_df_merged_all_path}")


✅ Saved ALL merged options data to ./options_data_yearwise/options_df_merged_all.parquet


In [14]:
#EXIT PRICE
def get_exit_prices_by_pocket(strikes_df_merged, options_df_merged_all):
    # Work on a copy but only keep index and Exit Option Price
    strikes = strikes_df_merged.copy()
    options = options_df_merged_all.copy()

    # Compute fields for processing (not assigned to strikes, used locally)
    exit_times = pd.to_datetime(strikes['Exit Time'])
    expiries = pd.to_datetime(strikes['Expiry'])
    strikes_flt = strikes['Strike'].astype(float)
    types = strikes['Option Type'].astype(str)

    options['DateTime'] = pd.to_datetime(options['DateTime'])
    options['ExpiryDate'] = pd.to_datetime(options['ExpiryDate'])
    options['StrikePrice'] = options['StrikePrice'].astype(float)
    options['Type'] = options['Type'].astype(str)

    # We'll compare expiry by DATE ONLY to avoid 15:15 vs 00:00 mismatches
    options['_ExpiryDateOnly'] = options['ExpiryDate'].dt.normalize()

    # Assign a pocket id by 3 month period (quarter) for grouping
    pockets = exit_times.dt.to_period('Q').astype(str)

    # Prepare output, NaN by default
    exit_option_prices = np.full(len(strikes), np.nan)

    # Add processing columns temporarily for grouping/index
    proc_df = pd.DataFrame({
        'ExitTime': exit_times,
        'ExpiryDate': expiries,
        'StrikePrice': strikes_flt,
        'Type': types,
        'Pocket': pockets
    }, index=strikes.index)

    for pocket, pocket_idx in proc_df.groupby('Pocket').groups.items():
        pocket_strikes = proc_df.loc[pocket_idx]
        print(f"Processing pocket: {pocket} ({len(pocket_strikes)} trades)")

        start = pocket_strikes['ExitTime'].min() - pd.Timedelta('2d')
        end = pocket_strikes['ExitTime'].max() + pd.Timedelta('2d')
        tick_chunk = options[(options['DateTime'] >= start) & (options['DateTime'] <= end)].copy()

        # For each trade in this pocket
        for trade_idx, trade in pocket_strikes.iterrows():
            mask = (
                (tick_chunk['ExpiryDate'] == trade['ExpiryDate']) &
                (np.isclose(tick_chunk['StrikePrice'], trade['StrikePrice'], atol=0.01)) &
                (tick_chunk['Type'] == trade['Type'])
            )
            candidates = tick_chunk[mask]
            if not candidates.empty:
                time_diffs = np.abs((candidates['DateTime'] - trade['ExitTime']).values.astype('timedelta64[s]'))
                min_idx = time_diffs.argmin()
                if time_diffs[min_idx] <= 60:
                    exit_option_prices[trade_idx] = candidates.iloc[min_idx]['Open']

    # Attach only the result column, keep all your original columns
    result = strikes_df_merged.copy()
    result['Exit Option Price'] = exit_option_prices
    return result

# Usage:
strikes_with_prices = get_exit_prices_by_pocket(strikes_df_merged, options_df_merged_all)

Processing pocket: 2021Q3 (5 trades)
Processing pocket: 2021Q4 (45 trades)
Processing pocket: 2022Q1 (44 trades)
Processing pocket: 2022Q2 (44 trades)
Processing pocket: 2022Q3 (54 trades)
Processing pocket: 2022Q4 (46 trades)
Processing pocket: 2023Q1 (43 trades)
Processing pocket: 2023Q2 (41 trades)
Processing pocket: 2023Q3 (49 trades)
Processing pocket: 2023Q4 (32 trades)
Processing pocket: 2024Q1 (45 trades)
Processing pocket: 2024Q2 (44 trades)
Processing pocket: 2024Q3 (42 trades)
Processing pocket: 2024Q4 (46 trades)
Processing pocket: 2025Q1 (50 trades)
Processing pocket: 2025Q2 (27 trades)


In [15]:
strikes_with_prices

,Entry Time,Exit Time,Date,Side,Entry Price,Exit Price,pnl_pts,exit_reason,Strike,Option Type,...,PivotHigh,A_distance,TP_mult_A,n_used,P_used,T,slippage,cost_pts,Side_trade,Exit Option Price
0,2021-09-24 13:35:00,2021-09-24 13:35:00,2021-09-24,Bearish,17846.00,17797.341667,48.158333,STOP,17950.0,CE,...,17796.841667,42.232750,1.0,16,0.28,20,0.5,0.5,short,80.70
1,2021-09-27 10:48:00,2021-09-27 12:55:00,2021-09-27,Bearish,17836.75,17884.025000,-47.775000,STOP,17900.0,CE,...,17883.525000,41.509125,1.0,16,0.28,20,0.5,0.5,short,97.25
2,2021-09-28 11:19:00,2021-09-28 11:52:00,2021-09-28,Bearish,17797.10,17782.699167,13.900833,TARGET,17850.0,CE,...,17873.200000,14.900833,1.0,18,0.10,20,0.5,0.5,short,67.15
3,2021-09-29 13:19:00,2021-09-29 13:19:00,2021-09-29,Bullish,17697.65,17743.891667,45.741667,STOP,17650.0,PE,...,17744.475000,16.282750,1.0,20,0.10,20,0.5,0.5,long,47.95
4,2021-09-30 13:25:00,2021-09-30 14:26:00,2021-09-30,Bearish,17626.05,17610.587500,14.962500,TARGET,17650.0,CE,...,17707.816667,15.962500,1.0,20,0.10,20,0.5,0.5,short,14.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
652,2025-06-05 12:11:00,2025-06-05 12:16:00,2025-06-05,Bullish,24846.55,24893.701464,46.651464,TARGET,24800.0,PE,...,24603.083333,47.651464,1.0,28,0.18,20,0.5,0.5,long,36.05
653,2025-06-06 10:34:00,2025-06-06 10:41:00,2025-06-06,Bullish,24861.05,24906.717429,45.167429,TARGET,24800.0,PE,...,24764.858333,46.167429,1.0,28,0.18,20,0.5,0.5,long,130.30
654,2025-06-11 11:54:00,2025-06-11 13:53:00,2025-06-11,Bullish,25215.40,25111.725000,-104.175000,STOP,25150.0,PE,...,25127.375000,45.993214,1.0,28,0.18,20,0.5,0.5,long,90.80
655,2025-06-12 10:51:00,2025-06-12 11:39:00,2025-06-12,Bearish,25057.20,25026.413111,30.286889,TARGET,25100.0,CE,...,25151.850000,31.286889,1.0,18,0.14,20,0.5,0.5,short,21.55


In [30]:
out = "/home/newberry3/disha/acd_pivot_roll_202106_202506/trades_df.xlsx"
trades_df.to_excel(out, index=False)


In [16]:
# ✅ Count NaNs in each column
nan_counts = strikes_with_prices.isna().sum()
print("NaN counts per column:")
print(nan_counts)

# ✅ Filter rows where ANY column has a NaN
nan_rows = strikes_with_prices[strikes_with_prices.isna().any(axis=1)]

print(f"\nRows with at least one NaN ({len(nan_rows)} rows) have been written to 'nan_rows.xlsx'.")

# Save to Excel
nan_rows.to_excel("nan_rows.xlsx", index=False)

NaN counts per column:
Entry Time           0
Exit Time            0
Date                 0
Side                 0
Entry Price          0
Exit Price           0
pnl_pts              0
exit_reason          0
Strike               0
Option Type          0
Expiry               0
Option Price         0
IV                   0
Delta                0
ORH                  0
ORL                  0
PivotLow             0
PivotHigh            0
A_distance           0
TP_mult_A            0
n_used               0
P_used               0
T                    0
slippage             0
cost_pts             0
Side_trade           0
Exit Option Price    0
dtype: int64

Rows with at least one NaN (0 rows) have been written to 'nan_rows.xlsx'.


In [40]:
trades_df=strikes_with_prices
trades_df = trades_df.rename(columns={"Entry Price": "Spot Entry Price"})
trades_df = trades_df.rename(columns={"Exit Price": "Spot Exit Price"})
trades_df = trades_df.rename(columns={"Option Price": "Entry Price"})
trades_df = trades_df.rename(columns={"Exit Option Price": "Exit Price"})
trades_df

,Entry Time,Exit Time,Date,Side,Spot Entry Price,Spot Exit Price,pnl_pts,exit_reason,Strike,Option Type,...,PivotHigh,A_distance,TP_mult_A,n_used,P_used,T,slippage,cost_pts,Side_trade,Exit Price
0,2021-09-24 13:35:00,2021-09-24 13:35:00,2021-09-24,Bearish,17846.00,17797.341667,48.158333,STOP,17950.0,CE,...,17796.841667,42.232750,1.0,16,0.28,20,0.5,0.5,short,80.70
1,2021-09-27 10:48:00,2021-09-27 12:55:00,2021-09-27,Bearish,17836.75,17884.025000,-47.775000,STOP,17900.0,CE,...,17883.525000,41.509125,1.0,16,0.28,20,0.5,0.5,short,97.25
2,2021-09-28 11:19:00,2021-09-28 11:52:00,2021-09-28,Bearish,17797.10,17782.699167,13.900833,TARGET,17850.0,CE,...,17873.200000,14.900833,1.0,18,0.10,20,0.5,0.5,short,67.15
3,2021-09-29 13:19:00,2021-09-29 13:19:00,2021-09-29,Bullish,17697.65,17743.891667,45.741667,STOP,17650.0,PE,...,17744.475000,16.282750,1.0,20,0.10,20,0.5,0.5,long,47.95
4,2021-09-30 13:25:00,2021-09-30 14:26:00,2021-09-30,Bearish,17626.05,17610.587500,14.962500,TARGET,17650.0,CE,...,17707.816667,15.962500,1.0,20,0.10,20,0.5,0.5,short,14.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
652,2025-06-05 12:11:00,2025-06-05 12:16:00,2025-06-05,Bullish,24846.55,24893.701464,46.651464,TARGET,24800.0,PE,...,24603.083333,47.651464,1.0,28,0.18,20,0.5,0.5,long,36.05
653,2025-06-06 10:34:00,2025-06-06 10:41:00,2025-06-06,Bullish,24861.05,24906.717429,45.167429,TARGET,24800.0,PE,...,24764.858333,46.167429,1.0,28,0.18,20,0.5,0.5,long,130.30
654,2025-06-11 11:54:00,2025-06-11 13:53:00,2025-06-11,Bullish,25215.40,25111.725000,-104.175000,STOP,25150.0,PE,...,25127.375000,45.993214,1.0,28,0.18,20,0.5,0.5,long,90.80
655,2025-06-12 10:51:00,2025-06-12 11:39:00,2025-06-12,Bearish,25057.20,25026.413111,30.286889,TARGET,25100.0,CE,...,25151.850000,31.286889,1.0,18,0.14,20,0.5,0.5,short,21.55


In [41]:
#TRANSACTION COSTS
# Lot sizes for reference (already in your code)
lot_sizes = {"NIFTY": 75, "BANKNIFTY": 30, "FINNIFTY": 40}
lot_multiplier = 1  # Use as per your actual qty scaling

# Spread
def get_spread(value):
    if 0 <= value <= 10:
        spread_percentage = 0.05
    else:
        spread_percentage = 0.1
    return (value * spread_percentage) / 100

# Charges calculation (as per your logic)
def calculate_option_trade_charges(buy_price, sell_price, lot_size, brokerage_per_order=0):
    etc_rate = 0.00035
    sebi_rate = 0.000001
    stamp_duty_rate = 0.00003
    stt_rate = 0.001
    gst_rate = 0.18

    turnover = (buy_price + sell_price) * lot_size
    etc = etc_rate * turnover
    sebi = sebi_rate * turnover
    stamp_duty = stamp_duty_rate * (buy_price * lot_size)
    stt = stt_rate * (sell_price * lot_size)
    brokerage = 2 * brokerage_per_order
    gst = gst_rate * (etc + sebi + brokerage)
    total_charges = etc + sebi + stamp_duty + stt + brokerage + gst
    gross_pnl = (sell_price - buy_price) * lot_size
    net_pnl = gross_pnl - total_charges

    return {
        'Turnover': turnover,
        'ETC': etc,
        'SEBI': sebi,
        'Stamp Duty': stamp_duty,
        'STT': stt,
        'Brokerage': brokerage,
        'GST': gst,
        'Total Charges': total_charges,
        'Gross P&L': gross_pnl,
        'Net P&L': net_pnl
    }

In [37]:
import pandas as pd
import numpy as np

# ===== 0) Expect your DataFrame already loaded as `trades_df` =====
# It should have columns shown in your sample:
# ['Entry Time','Exit Time','Date','Side','Spot Entry Price','Spot Exit Price','pnl_pts','exit_reason',
#  'Strike','Option Type','Expiry','Entry Price','IV','Delta','ORH','ORL','PivotLow','PivotHigh',
#  'A_distance','TP_mult_A','n_used','P_used','T','slippage','cost_pts','Side_trade','Exit Price']

# --- Helpers to parse numbers that include commas ---
def _to_float(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)): return float(x)
    return float(str(x).replace(',', ''))

# --- 1) Normalize types / clean numbers ---
trades_df = trades_df.copy()

# Time columns
trades_df["Entry Time"] = pd.to_datetime(trades_df["Entry Time"])
trades_df["Exit Time"]  = pd.to_datetime(trades_df["Exit Time"])

# Clean numeric columns that may contain commas (spot and option prices, strike, etc.)
num_cols_maybe_comma = [
    "Spot Entry Price","Spot Exit Price","Entry Price","Exit Price","Strike",
    "pnl_pts","ORH","ORL","PivotLow","PivotHigh","A_distance","IV","Delta",
    "TP_mult_A","n_used","P_used","T","slippage","cost_pts"
]
for c in num_cols_maybe_comma:
    if c in trades_df.columns:
        trades_df[c] = trades_df[c].apply(_to_float)

# --- 2) Option premiums (entry/exit) ---
# Use your column names directly
if not {"Entry Price","Exit Price"}.issubset(trades_df.columns):
    raise ValueError("Expected columns 'Entry Price' (option entry premium) and 'Exit Price' (option exit premium).")

trades_df["OptEntryPremium"] = trades_df["Entry Price"].astype(float)
trades_df["OptExitPremium"]  = trades_df["Exit Price"].astype(float)

# --- 3) Lot size (you must define `lot_sizes` and `lot_multiplier` beforehand) ---
index_type = "NIFTY"  # change if needed
lot_size = lot_sizes[index_type]
final_lot_size = lot_size * lot_multiplier

# --- 4) Apply your spread model to option premiums (you must define get_spread(premium)) ---
trades_df["Entry_Spread"] = trades_df["OptEntryPremium"].apply(get_spread)
trades_df["Exit_Spread"]  = trades_df["OptExitPremium"].apply(get_spread)

# --- 5) Direction-aware adjusted prices (defaults to SHORT if Side_trade missing) ---
def adjusted_prices(row):
    pos = (row.get("Side_trade") or "").strip().lower()  # expected 'short' or 'long'
    if pos not in ("long","short"):
        pos = "short"  # default if not provided

    e  = float(row["OptEntryPremium"])
    x  = float(row["OptExitPremium"])
    es = float(row["Entry_Spread"])
    xs = float(row["Exit_Spread"])

    if pos == "long":
        # buy at (entry + spread), sell at (exit - spread)
        adj_entry, adj_exit = e + es, x - xs
        buy_price, sell_price = adj_entry, adj_exit
    else:
        # short: sell at (entry - spread), buy back at (exit + spread)
        adj_entry, adj_exit = e - es, x + xs
        buy_price, sell_price = adj_exit, adj_entry

    return pd.Series({
        "Position": pos,
        "Adj_Entry_Price": round(adj_entry, 2),
        "Adj_Exit_Price":  round(adj_exit, 2),
        # For brokerage/fees calc (your function expects explicit buy & sell)
        "BuyPx_forCharges":  round(buy_price, 2),
        "SellPx_forCharges": round(sell_price, 2),
    })

ap = trades_df.apply(adjusted_prices, axis=1)
trades_df = pd.concat([trades_df, ap], axis=1)

# --- 6) Net P&L with charges (you must define calculate_option_trade_charges) ---
def compute_net_leg_pnl(buy_px, sell_px, lot):
    res = calculate_option_trade_charges(buy_price=buy_px, sell_price=sell_px, lot_size=lot)
    return float(res.get("Net P&L", np.nan))

trades_df["Net_PnL_With_Spread"] = trades_df.apply(
    lambda r: compute_net_leg_pnl(r["BuyPx_forCharges"], r["SellPx_forCharges"], final_lot_size),
    axis=1
)

# --- 7) Gross P&L (no spreads/fees), respecting direction ---
def compute_gross_pnl(row):
    entry = float(row["OptEntryPremium"])
    exit_ = float(row["OptExitPremium"])
    if row["Position"] == "long":
        return (exit_ - entry) * final_lot_size
    else:
        return (entry - exit_) * final_lot_size

trades_df["Gross_PnL"] = trades_df.apply(compute_gross_pnl, axis=1)

# --- 8) Add year/date keys & sort for analytics like your template ---
trades_df["Year"] = trades_df["Exit Time"].dt.year
trades_df["Date"] = trades_df["Exit Time"].dt.date
trades_df = trades_df.sort_values("Exit Time").reset_index(drop=True)

# === 9) Analytics summary (unchanged structure from your template) ===
def calculate_drawdown(pnl_series):
    cumulative = pnl_series.cumsum()
    high_watermark = cumulative.cummax()
    drawdown = cumulative - high_watermark
    max_dd = drawdown.min()
    max_dd_pct = max_dd / high_watermark.max() if high_watermark.max() != 0 else 0
    return max_dd, max_dd_pct

def sortino_ratio(returns, periods):
    mean_return = np.mean(returns)
    downside = returns[returns < 0]
    downside_deviation = np.std(downside, ddof=1) if len(downside) > 0 else 1e-9
    return mean_return / downside_deviation * np.sqrt(periods)

last_date = trades_df["Exit Time"].max()
years = sorted(trades_df["Year"].unique())
periods = {f"Year {y}": [pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y+1}-01-01")] for y in years}
periods.update({
    "Last 6 Months": [last_date - pd.DateOffset(months=6), last_date],
    "Last 1 Year":   [last_date - pd.DateOffset(years=1), last_date],
    "Last 2 Years":  [last_date - pd.DateOffset(years=2), last_date],
    "Full Period":   [trades_df["Exit Time"].min(), trades_df["Exit Time"].max()]
})

summary_rows = []
for period_name, (start_time, end_time) in periods.items():
    chunk = trades_df[(trades_df["Exit Time"] >= start_time) & (trades_df["Exit Time"] < end_time)].copy()
    if chunk.empty: 
        continue

    chunk["Date"] = chunk["Exit Time"].dt.date
    daily_df = (chunk.groupby("Date")
                .agg(Net_PnL_With_Spread=("Net_PnL_With_Spread","sum"),
                     Gross_PnL=("Gross_PnL","sum"))
                .reset_index())

    denom = daily_df["Net_PnL_With_Spread"].abs().sum()
    daily_df["Daily_Return"] = daily_df["Net_PnL_With_Spread"] / (denom if denom != 0 else 1)

    returns = daily_df["Daily_Return"].fillna(0)
    win_rate = (chunk["Net_PnL_With_Spread"] > 0).mean()
    max_dd, max_dd_pct = calculate_drawdown(daily_df["Net_PnL_With_Spread"])
    sortino_12 = sortino_ratio(returns, 12)
    sortino_48 = sortino_ratio(returns, 48)
    total_return = daily_df["Net_PnL_With_Spread"].sum()
    annualized_return = (1 + returns.mean()) ** 252 - 1 if len(returns) > 0 else np.nan

    wins = chunk.loc[chunk["Net_PnL_With_Spread"] > 0, "Net_PnL_With_Spread"]
    losses = chunk.loc[chunk["Net_PnL_With_Spread"] < 0, "Net_PnL_With_Spread"]
    std_win = wins.std(ddof=1) if not wins.empty else np.nan
    std_loss = losses.std(ddof=1) if not losses.empty else np.nan
    avg_win = wins.mean() if not wins.empty else np.nan
    avg_loss = losses.mean() if not losses.empty else np.nan

    # Split by 'Side' string in your df (Bullish/Bearish)
    chunk_bull = chunk[chunk["Side"].str.lower() == "bullish"]
    bullish_pnl = chunk_bull["Net_PnL_With_Spread"].sum()
    bullish_win = (chunk_bull["Net_PnL_With_Spread"] > 0).mean() if len(chunk_bull) > 0 else np.nan
    bull_dd, bull_dd_pct = calculate_drawdown(
        chunk_bull.groupby("Date")["Net_PnL_With_Spread"].sum() if not chunk_bull.empty else pd.Series([0])
    )

    chunk_bear = chunk[chunk["Side"].str.lower() == "bearish"]
    bearish_pnl = chunk_bear["Net_PnL_With_Spread"].sum()
    bearish_win = (chunk_bear["Net_PnL_With_Spread"] > 0).mean() if len(chunk_bear) > 0 else np.nan
    bear_dd, bear_dd_pct = calculate_drawdown(
        chunk_bear.groupby("Date")["Net_PnL_With_Spread"].sum() if not chunk_bear.empty else pd.Series([0])
    )

    summary_rows.append({
        "Period": period_name,
        "Total Net PnL (With Spread)": total_return,
        "Total Gross PnL": daily_df["Gross_PnL"].sum(),
        "Bullish PnL": bullish_pnl,
        "Bullish Win %": round(bullish_win*100,2) if pd.notnull(bullish_win) else np.nan,
        "Bullish Max DD": bull_dd,
        "Bullish Max DD %": bull_dd_pct * 100,
        "Bearish PnL": bearish_pnl,
        "Bearish Win %": round(bearish_win*100,2) if pd.notnull(bearish_win) else np.nan,
        "Bearish Max DD": bear_dd,
        "Bearish Max DD %": bear_dd_pct * 100,
        "Win Rate": round(win_rate*100,2),
        "Sortino (12)": sortino_12,
        "Sortino (48)": sortino_48,
        "Max Drawdown": max_dd,
        "Max Drawdown %": max_dd_pct * 100,
        "Annualized Return %": round(annualized_return*100,2) if not pd.isnull(annualized_return) else np.nan,
        "Mean Daily Return": returns.mean(),
        "Std Dev Daily Return": returns.std(),
        "Std Win": std_win,
        "Std Loss": std_loss,
        "Avg Win": avg_win,
        "Avg Loss": avg_loss
    })

summary_df = pd.DataFrame(summary_rows)

In [42]:
import pandas as pd
import numpy as np

# ===== Expect your DataFrame as `trades_df` with columns shown in your sample =====

# --- Helpers ---
def _to_float(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)): return float(x)
    return float(str(x).replace(',', ''))

# 1) Normalize
trades_df = trades_df.copy()
trades_df["Entry Time"] = pd.to_datetime(trades_df["Entry Time"])
trades_df["Exit Time"]  = pd.to_datetime(trades_df["Exit Time"])

num_cols_maybe_comma = [
    "Spot Entry Price","Spot Exit Price","Entry Price","Exit Price","Strike",
    "pnl_pts","ORH","ORL","PivotLow","PivotHigh","A_distance","IV","Delta",
    "TP_mult_A","n_used","P_used","T","slippage","cost_pts"
]
for c in num_cols_maybe_comma:
    if c in trades_df.columns:
        trades_df[c] = trades_df[c].apply(_to_float)

# 2) Force premium-selling mapping: Bullish -> short PE, Bearish -> short CE
m = trades_df["Side"].str.lower().fillna("")
trades_df.loc[m=="bullish", ["Option Type","Side_trade"]] = ["PE","short"]
trades_df.loc[m=="bearish", ["Option Type","Side_trade"]] = ["CE","short"]

# 3) Option premiums
if not {"Entry Price","Exit Price"}.issubset(trades_df.columns):
    raise ValueError("Expected 'Entry Price' (option entry premium) and 'Exit Price' (option exit premium).")

trades_df["OptEntryPremium"] = trades_df["Entry Price"].astype(float)
trades_df["OptExitPremium"]  = trades_df["Exit Price"].astype(float)

# 4) Lot size (define these in your env)
index_type = "NIFTY"              # change if needed
lot_size = lot_sizes[index_type]  # e.g., {"NIFTY": 50, ...}
final_lot_size = lot_size * lot_multiplier

# 5) Spreads (define get_spread(premium) in your env)
trades_df["Entry_Spread"] = trades_df["OptEntryPremium"].apply(get_spread)
trades_df["Exit_Spread"]  = trades_df["OptExitPremium"].apply(get_spread)

# 6) Adjusted prices for SHORT positions (we enforce short above)
#    Short: sell at (entry - spread), buy back at (exit + spread)
def adjusted_prices_short(row):
    e  = float(row["OptEntryPremium"])
    x  = float(row["OptExitPremium"])
    es = float(row["Entry_Spread"])
    xs = float(row["Exit_Spread"])

    adj_entry = e - es  # sell
    adj_exit  = x + xs  # buy
    # for charges function (expects explicit buy & sell):
    buy_px, sell_px = adj_exit, adj_entry

    return pd.Series({
        "Position": "short",
        "Adj_Entry_Price": round(adj_entry, 2),
        "Adj_Exit_Price":  round(adj_exit, 2),
        "BuyPx_forCharges":  round(buy_px, 2),
        "SellPx_forCharges": round(sell_px, 2),
    })

trades_df = pd.concat([trades_df, trades_df.apply(adjusted_prices_short, axis=1)], axis=1)

# 7) Net P&L with charges (define calculate_option_trade_charges in your env)
def compute_net_leg_pnl(buy_px, sell_px, lot):
    res = calculate_option_trade_charges(buy_price=buy_px, sell_price=sell_px, lot_size=lot)
    return float(res.get("Net P&L", np.nan))

trades_df["Net_PnL_With_Spread"] = trades_df.apply(
    lambda r: compute_net_leg_pnl(r["BuyPx_forCharges"], r["SellPx_forCharges"], final_lot_size),
    axis=1
)

# 8) Gross P&L (no spreads/fees) for SHORT = (entry - exit) * lot
trades_df["Gross_PnL"] = (trades_df["OptEntryPremium"] - trades_df["OptExitPremium"]) * final_lot_size

# 9) Keys & sort (for downstream analytics)
trades_df["Year"] = trades_df["Exit Time"].dt.year
trades_df["Date"] = trades_df["Exit Time"].dt.date
trades_df = trades_df.sort_values("Exit Time").reset_index(drop=True)

# === Optional: build the same summary as before ===
def calculate_drawdown(pnl_series):
    cumulative = pnl_series.cumsum()
    high_watermark = cumulative.cummax()
    drawdown = cumulative - high_watermark
    max_dd = drawdown.min()
    max_dd_pct = max_dd / high_watermark.max() if high_watermark.max() != 0 else 0
    return max_dd, max_dd_pct

def sortino_ratio(returns, periods):
    mean_return = np.mean(returns)
    downside = returns[returns < 0]
    downside_deviation = np.std(downside, ddof=1) if len(downside) > 0 else 1e-9
    return mean_return / downside_deviation * np.sqrt(periods)

last_date = trades_df["Exit Time"].max()
years = sorted(trades_df["Year"].unique())
periods = {f"Year {y}": [pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y+1}-01-01")] for y in years}
periods.update({
    "Last 6 Months": [last_date - pd.DateOffset(months=6), last_date],
    "Last 1 Year":   [last_date - pd.DateOffset(years=1), last_date],
    "Last 2 Years":  [last_date - pd.DateOffset(years=2), last_date],
    "Full Period":   [trades_df["Exit Time"].min(), trades_df["Exit Time"].max()]
})

summary_rows = []
for period_name, (start_time, end_time) in periods.items():
    chunk = trades_df[(trades_df["Exit Time"] >= start_time) & (trades_df["Exit Time"] < end_time)].copy()
    if chunk.empty:
        continue

    chunk["Date"] = chunk["Exit Time"].dt.date
    daily_df = (chunk.groupby("Date")
                .agg(Net_PnL_With_Spread=("Net_PnL_With_Spread","sum"),
                     Gross_PnL=("Gross_PnL","sum"))
                .reset_index())

    denom = daily_df["Net_PnL_With_Spread"].abs().sum()
    daily_df["Daily_Return"] = daily_df["Net_PnL_With_Spread"] / (denom if denom != 0 else 1)

    returns = daily_df["Daily_Return"].fillna(0)
    win_rate = (chunk["Net_PnL_With_Spread"] > 0).mean()
    max_dd, max_dd_pct = calculate_drawdown(daily_df["Net_PnL_With_Spread"])
    sortino_12 = sortino_ratio(returns, 12)
    sortino_48 = sortino_ratio(returns, 48)
    total_return = daily_df["Net_PnL_With_Spread"].sum()
    annualized_return = (1 + returns.mean()) ** 252 - 1 if len(returns) > 0 else np.nan

    wins = chunk.loc[chunk["Net_PnL_With_Spread"] > 0, "Net_PnL_With_Spread"]
    losses = chunk.loc[chunk["Net_PnL_With_Spread"] < 0, "Net_PnL_With_Spread"]
    std_win = wins.std(ddof=1) if not wins.empty else np.nan
    std_loss = losses.std(ddof=1) if not losses.empty else np.nan
    avg_win = wins.mean() if not wins.empty else np.nan
    avg_loss = losses.mean() if not losses.empty else np.nan

    # Split by Side string (Bullish/Bearish)
    chunk_bull = chunk[chunk["Side"].str.lower() == "bullish"]
    bull_pnl = chunk_bull["Net_PnL_With_Spread"].sum()
    bull_win = (chunk_bull["Net_PnL_With_Spread"] > 0).mean() if len(chunk_bull) > 0 else np.nan
    bull_dd, bull_dd_pct = calculate_drawdown(
        chunk_bull.groupby("Date")["Net_PnL_With_Spread"].sum() if not chunk_bull.empty else pd.Series([0])
    )

    chunk_bear = chunk[chunk["Side"].str.lower() == "bearish"]
    bear_pnl = chunk_bear["Net_PnL_With_Spread"].sum()
    bear_win = (chunk_bear["Net_PnL_With_Spread"] > 0).mean() if len(chunk_bear) > 0 else np.nan
    bear_dd, bear_dd_pct = calculate_drawdown(
        chunk_bear.groupby("Date")["Net_PnL_With_Spread"].sum() if not chunk_bear.empty else pd.Series([0])
    )

    summary_rows.append({
        "Period": period_name,
        "Total Net PnL (With Spread)": total_return,
        "Total Gross PnL": daily_df["Gross_PnL"].sum(),
        "Bullish PnL": bull_pnl,
        "Bullish Win %": round(bull_win*100,2) if pd.notnull(bull_win) else np.nan,
        "Bullish Max DD": bull_dd,
        "Bullish Max DD %": bull_dd_pct * 100,
        "Bearish PnL": bear_pnl,
        "Bearish Win %": round(bear_win*100,2) if pd.notnull(bear_win) else np.nan,
        "Bearish Max DD": bear_dd,
        "Bearish Max DD %": bear_dd_pct * 100,
        "Win Rate": round(win_rate*100,2),
        "Sortino (12)": sortino_12,
        "Sortino (48)": sortino_48,
        "Max Drawdown": max_dd,
        "Max Drawdown %": max_dd_pct * 100,
        "Annualized Return %": round(annualized_return*100,2) if not pd.isnull(annualized_return) else np.nan,
        "Mean Daily Return": returns.mean(),
        "Std Dev Daily Return": returns.std(),
        "Std Win": std_win,
        "Std Loss": std_loss,
        "Avg Win": avg_win,
        "Avg Loss": avg_loss
    })

summary_df = pd.DataFrame(summary_rows)


In [43]:
from pathlib import Path

# === Save Excel: All Trades + Summary ===
out_path = Path("/home/newberry3/disha/acd_pivot_roll_202106_202506/analytics_report_2.xlsx")  # change if needed
with pd.ExcelWriter(out_path, engine="xlsxwriter", datetime_format="yyyy-mm-dd hh:mm", date_format="yyyy-mm-dd") as writer:
    trades_df.to_excel(writer, index=False, sheet_name="All_Trades")
    summary_df.to_excel(writer, index=False, sheet_name="Analytics_Summary")
print(f"✅ Analytics Excel saved to: {out_path}")


✅ Analytics Excel saved to: /home/newberry3/disha/acd_pivot_roll_202106_202506/analytics_report_2.xlsx
